# Reconstructing chiller operating schedules

**Main analysis** · 15-minute data · Seven-system leave-one-system-out evaluation

This notebook implements reference-state construction, contextual H/V/G features, event-aware neural-network training, held-out inference, and daily/monthly schedule evaluation. At inference, the model uses aggregate electricity, weather, and calendar context; measured chiller electricity is used only for supervised reference labels.

Use authorized local input files as documented in [`DATA_SCHEMA.md`](../DATA_SCHEMA.md). Run this notebook before `02_validation_extensions.ipynb`. Saved outputs are intentionally excluded from the repository.

In [ ]:
# CELL 1 — Imports, paths, runtime, and FINAL configuration
import os, gc, json, math, time, random, warnings, copy, itertools, shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.preprocessing import SplineTransformer, StandardScaler, RobustScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    roc_auc_score, average_precision_score,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from IPython.display import display

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

@dataclass
class Config:
    task_dir_candidates: tuple = (os.environ.get('TASK1_WORKDIR', '.'),)
    load_file: str = 'data/meter_data.xlsx'
    load_sheet: str = 'meter_data'
    weather_file: str = 'data/weather_data.xlsx'
    weather_sheet: str = 'weather'
    result_dir: str = 'results'
    reuse_previous_compatible_checkpoints: bool = True
    previous_compatible_result_dir: str = 'previous_models'

    # Grid / input QC
    dt_minutes: int = 15
    steps_per_day: int = 96
    ptotal_max_interp_gap_steps: int = 2
    weather_max_interp_gap_steps: int = 4
    min_original_day_ptotal_coverage: float = 0.90
    min_original_day_weather_coverage: float = 0.90
    min_context_days_per_month: int = 15
    min_horizontal_context_fraction: float = 0.75
    min_vertical_context_days: int = 15

    # One causal weather scalar over [t-4h,t)
    weather_backward_hours: int = 4
    weather_backward_steps: int = 16
    weather_backward_min_periods: int = 16

    # Feature clipping
    robust_z_clip: float = 5.0
    delta_clip: float = 5.0
    ratio_clip_low: float = -10.0
    ratio_clip_high: float = 10.0

    # MLP
    hidden1: int = 64
    hidden2: int = 32
    hidden3: int = 16
    dropout: float = 0.05

    # Operational-event semantics
    decision_threshold: float = 0.50
    event_pre_window_min: int = 60
    event_pre_dominance_ratio: float = 0.75
    event_pre_min_valid_fraction: float = 0.75
    event_post_steps: int = 2
    morning_start: str = '04:00'
    morning_end: str = '10:00'
    day_status_min_coverage: float = 0.90
    min_truth_day_coverage: float = 0.90

    # Curriculum:
    # (LR, ordinary, sec±30, sec±15, sec exact, pri±30, pri±15, pri exact)
    curriculum: tuple = (
        (1.0e-3, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00),
        (5.0e-4, 1.00, 1.10, 1.30, 1.75, 1.40, 2.25, 4.00),
        (2.0e-4, 1.00, 1.25, 1.75, 2.75, 1.75, 3.50, 7.00),
        (1.0e-4, 1.00, 1.40, 2.25, 3.75, 2.50, 5.00, 10.00),
        (5.0e-5, 1.00, 1.50, 2.50, 4.00, 3.00, 6.00, 12.00),
    )

    # Fixed event-aware monitoring objective
    fixed_weight_ordinary: float = 1.0
    fixed_weight_secondary_pm30: float = 1.5
    fixed_weight_secondary_pm15: float = 2.5
    fixed_weight_secondary_exact: float = 4.0
    fixed_weight_primary_pm30: float = 3.0
    fixed_weight_primary_pm15: float = 6.0
    fixed_weight_primary_exact: float = 12.0

    # Training
    # Stages 1-4 are curriculum/warm-up stages only.
    # Every seed MUST reach Stage 5.
    # Only Stage-5 checkpoints are eligible for final model selection.
    max_epochs: int = 500
    min_epochs_per_stage: int = 8
    max_epochs_per_stage: int = 60
    train_plateau_patience: int = 6
    val_rise_patience: int = 5
    final_stage_patience: int = 10
    smoothing_window: int = 3
    train_plateau_min_rel_improvement: float = 0.001
    selection_gap_lambda: float = 0.15
    batch_size: int = 8192
    eval_batch_size: int = 32768
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    parallel_seed_training: bool = True
    use_amp: bool = True

    # LOBO
    lobo_seeds: tuple = (42, 123, 456, 789, 2026)
    ensemble_min_votes: int = 3
    max_folds: Optional[int] = None
    run_training: bool = True
    resume_existing: bool = True
    rebuild_truth: bool = False
    rebuild_features: bool = True

    # Evaluation
    min_month_interval_coverage: float = 0.90
    min_month_truth_coverage: float = 0.90
    timing_tolerance_15_min: int = 15
    timing_tolerance_30_min: int = 30

    # Paper outputs
    save_models: bool = True
    save_plots: bool = True
    plot_dpi: int = 220
    representative_days_per_building: int = 2
    save_all_monthly_schedule_plots: bool = True

CFG = Config()

assert CFG.weather_backward_steps == int(CFG.weather_backward_hours * 60 / CFG.dt_minutes)
assert CFG.weather_backward_min_periods == CFG.weather_backward_steps
assert CFG.event_pre_window_min % CFG.dt_minutes == 0
assert len(CFG.curriculum[0]) == 8
assert len(CFG.curriculum) == 5
assert CFG.max_epochs_per_stage >= CFG.min_epochs_per_stage
assert CFG.final_stage_patience >= 1
assert CFG.decision_threshold == 0.50
assert CFG.min_truth_day_coverage >= 0.90
assert CFG.min_month_truth_coverage >= 0.90
assert CFG.max_epochs > (
    (len(CFG.curriculum) - 1) * CFG.max_epochs_per_stage
    + CFG.min_epochs_per_stage
)

def first_existing_dir(paths):
    for p in paths:
        q = Path(p)
        if q.exists():
            return q
    return Path(paths[0])

TASK_DIR = first_existing_dir(CFG.task_dir_candidates)
LOAD_FILE = TASK_DIR / CFG.load_file
WEATHER_FILE = TASK_DIR / CFG.weather_file
RESULT_DIR = TASK_DIR / CFG.result_dir

TRUTH_DIR = RESULT_DIR / 'truth'
FEATURE_DIR = RESULT_DIR / 'features'
MODEL_DIR = RESULT_DIR / 'models'
TABLE_DIR = RESULT_DIR / 'tables'
PRED_DIR = RESULT_DIR / 'predictions'
PLOT_DIR = RESULT_DIR / 'figures'
PLOT_DAILY_DIR = PLOT_DIR / 'daily_reconstruction'
PLOT_MONTHLY_DIR = PLOT_DIR / 'monthly_schedule'
PLOT_PAPER_DIR = PLOT_DIR / 'paper_ready'
PLOT_TRAIN_DIR = PLOT_DIR / 'training'

for p in [
    RESULT_DIR, TRUTH_DIR, FEATURE_DIR, MODEL_DIR, TABLE_DIR, PRED_DIR,
    PLOT_DIR, PLOT_DAILY_DIR, PLOT_MONTHLY_DIR, PLOT_PAPER_DIR, PLOT_TRAIN_DIR
]:
    p.mkdir(parents=True, exist_ok=True)


# The decoder change is post-training only. Reuse the immediately preceding
# eventrulefix_v3 Stage-5 checkpoints when they exist, while keeping all new
# final paper outputs in this clean fixed-threshold result directory.
if CFG.reuse_previous_compatible_checkpoints:
    previous_model_dir = (
        TASK_DIR
        / CFG.previous_compatible_result_dir
        / 'models'
    )

    final_has_checkpoints = any(
        MODEL_DIR.rglob('seed_*_best.pt')
    )

    if (
        previous_model_dir.exists()
        and not final_has_checkpoints
    ):
        shutil.copytree(
            previous_model_dir,
            MODEL_DIR,
            dirs_exist_ok=True,
        )

        print(
            'Copied previous compatible Stage-5 checkpoints from:',
            previous_model_dir,
        )

FREQ_MIN = CFG.dt_minutes
EXPECTED_DAILY_SAMPLES = CFG.steps_per_day
DATETIME_COL = 'DateTime_local'
WEATHER_DATETIME_COL = 'DateTime_local'
TEMPERATURE_COL = 'outdoor_temperature'
DIRECT_RADIATION_COL = 'direct_radiation'
SHORTWAVE_RADIATION_COL = 'shortwave_radiation'
DIRECT_RADIATION_COMPLETENESS_THRESHOLD = 0.95
SEASONS = {}

print('Task directory:', TASK_DIR)
print('Result directory:', RESULT_DIR)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print(json.dumps(asdict(CFG), indent=2, default=str))

In [ ]:
# CELL 2 — Model constants and seven-system experiment set
STATE_MIN_PERSISTENCE_STEPS = 2
ACTIVE_SEED_MIN_PERSISTENCE_STEPS = 4

STTR_N_SPLITS = 5
STTR_SPLINE_KNOTS = 5
STTR_SPLINE_DEGREE = 3
STTR_RIDGE_ALPHA = 3.0
STTR_TARGET_LOW_Q = 0.05
STTR_TARGET_HIGH_Q = 0.95
STTR_FLOOR_Q = 0.10
STTR_CEILING_Q = 0.95
STTR_SMOOTH_HALFLIFE_STEPS = 6
STTR_MIN_TRAIN_ACTIVE_SAMPLES = 30

ALL_BUILDING_NAMES = [f'S{i}' for i in range(1, 8)]
BUILDING_NAMES = list(ALL_BUILDING_NAMES)

assert len(BUILDING_NAMES) == 7
assert len(set(BUILDING_NAMES)) == 7
assert all(b in ALL_BUILDING_NAMES for b in BUILDING_NAMES)

# Optional legacy/fallback parameters are supplied locally rather than
# publishing values associated with individual research systems.
# The primary percentage-based reference and the default ON/OFF logic do not
# use either fallback (reference_method='percentage', override disabled).
_settings_file = TASK_DIR / 'data/system_settings.json'
_system_settings = json.loads(_settings_file.read_text(encoding='utf-8')) if _settings_file.exists() else {}
LEGACY_THRESHOLDS = {
    b: _system_settings.get(b, {}).get('legacy_fixed_threshold_kw') for b in BUILDING_NAMES
}
NEAR_ZERO_LIMITS = {
    b: _system_settings.get(b, {}).get('near_zero_max_kw') for b in BUILDING_NAMES
}

def building_cfg(name):
    return {
        'building': name,
        'enabled': True,
        'total': f'total_load_{name}',
        'chiller': f'chiller_load_{name}',
        'reference_method': 'percentage',
        'off_max_ratio': 0.05,
        'aux_max_ratio': 0.20,
        'part_max_ratio': 0.60,
        'active_seed_quantile': 0.70,
        'use_ambiguous': False,
        'ambiguous_margin': 0.05,
        'use_absolute_near_zero': False,
        'near_zero_max_kw': NEAR_ZERO_LIMITS[name],
        'legacy_fixed_threshold_kw': LEGACY_THRESHOLDS[name],
    }

BUILDINGS = {b: building_cfg(b) for b in BUILDING_NAMES}
print('Seven-system experiment set:', BUILDING_NAMES)

## Reference-state construction

Measured chiller electricity defines the evaluation reference. The original multistate procedure distinguishes OFF, auxiliary, partial active, and high active operation; the latter two states form the binary ON label. Reference labels are never included in inference features.

In [ ]:
# CELL 3 — Weather features, active seed pool, and V5 STTR active reference

# -------------------------------------------------------------------------
# V5 CHANGE — only slow thermal-history features are used by STTR.
# -------------------------------------------------------------------------
REFERENCE_FEATURES = [
    "sttr_temp_mean_1h",
    "sttr_temp_mean_3h",
    "sttr_temp_mean_6h",
    "sttr_rad_sum_3h",
    "sttr_rad_sum_6h",
]


def add_weather_features(weather: pd.DataFrame, radiation_col: str) -> pd.DataFrame:
    """
    Build the slow thermal features used by the frozen V5 STTR method.

    The original Temperature and selected Radiation columns are retained in
    the returned DataFrame because they are also used by the daily plots and
    the ACTIVE-PART direct-weather plausibility check.
    """
    out = weather.copy()

    # V5 CHANGE — slow Temperature history.
    out["sttr_temp_mean_1h"] = (
        out[TEMPERATURE_COL].rolling(4, min_periods=2).mean()
    )
    out["sttr_temp_mean_3h"] = (
        out[TEMPERATURE_COL].rolling(12, min_periods=6).mean()
    )
    out["sttr_temp_mean_6h"] = (
        out[TEMPERATURE_COL].rolling(24, min_periods=12).mean()
    )

    # V5 CHANGE — slow Radiation history.
    out["sttr_rad_sum_3h"] = (
        out[radiation_col].rolling(12, min_periods=6).sum()
    )
    out["sttr_rad_sum_6h"] = (
        out[radiation_col].rolling(24, min_periods=12).sum()
    )

    return out


def sustained_true_mask(mask: pd.Series, min_steps: int) -> pd.Series:
    """Keep every point belonging to a True run with at least min_steps samples."""
    mask = mask.fillna(False).astype(bool)
    groups = (mask != mask.shift()).cumsum()
    run_size = mask.groupby(groups).transform("size")
    return mask & (run_size >= min_steps)


def build_active_seed_pool(
    df: pd.DataFrame,
    chiller_col: str,
    weather_features: pd.DataFrame,
    seed_quantile: float = 0.70,
    min_persistence_steps: int = ACTIVE_SEED_MIN_PERSISTENCE_STEPS,
) -> Tuple[pd.DataFrame, dict]:
    """
    Build the same conservative high-confidence active seed used in V4.

    V5 CHANGE:
    The seed itself is intentionally unchanged. Only the method that turns
    this seed into the weather-adjusted active reference has changed.
    """
    joined = df[[chiller_col]].join(
        weather_features[REFERENCE_FEATURES],
        how="left",
    )

    positive = joined[chiller_col].where(
        joined[chiller_col] > 0
    ).dropna()

    if positive.empty:
        raise ValueError(
            "No positive Chiller Load values are available for the active seed pool."
        )

    seed_threshold = float(
        positive.quantile(seed_quantile)
    )

    seed_candidate = joined[chiller_col].ge(
        seed_threshold
    )

    seed_mask = sustained_true_mask(
        seed_candidate,
        min_persistence_steps,
    )

    pool = joined.loc[
        seed_mask
    ].dropna(
        subset=[chiller_col] + REFERENCE_FEATURES
    ).copy()

    if len(pool) < STTR_MIN_TRAIN_ACTIVE_SAMPLES:
        raise ValueError(
            f"Active seed pool has only {len(pool)} complete samples. "
            "There are not enough high-confidence active samples for STTR."
        )

    hour = pool.index.hour
    night_like = (
        (hour >= 21)
        | (hour < 4)
    )

    meta = {
        "seed_threshold_kw": seed_threshold,
        "seed_quantile": seed_quantile,
        "seed_pool_samples": len(pool),
        "seed_pool_night_samples": int(night_like.sum()),
        "seed_pool_day_samples": int((~night_like).sum()),
        "seed_pool_median_kw": float(pool[chiller_col].median()),
        "seed_pool_q10_kw": float(pool[chiller_col].quantile(STTR_FLOOR_Q)),
        "seed_pool_q95_kw": float(pool[chiller_col].quantile(STTR_CEILING_Q)),
    }

    return pool, meta


def _make_sttr_model() -> Pipeline:
    """
    Smooth additive nonlinear weather-response model.

    SplineTransformer creates smooth basis functions separately for each
    thermal feature. Ridge regularization prevents an overly flexible trend.
    """
    return Pipeline(
        steps=[
            (
                "spline",
                SplineTransformer(
                    n_knots=STTR_SPLINE_KNOTS,
                    degree=STTR_SPLINE_DEGREE,
                    include_bias=False,
                ),
            ),
            (
                "scale",
                StandardScaler(),
            ),
            (
                "ridge",
                Ridge(alpha=STTR_RIDGE_ALPHA),
            ),
        ]
    )


def _shared_calendar_day_folds(
    target_index: pd.DatetimeIndex,
    pool_index: pd.DatetimeIndex,
    n_splits: int,
) -> Tuple[pd.Series, pd.Series]:
    """
    Assign target and seed samples with the SAME calendar-date -> fold map.

    V5 CHANGE / IMPORTANT:
    The same date must always have the same fold in target and active seed.
    This guarantees that a target day's active samples cannot leak into the
    model used to predict that day.
    """
    all_dates = pd.DatetimeIndex(
        sorted(pd.unique(target_index.normalize()))
    )

    date_to_fold = {
        date_value: i % n_splits
        for i, date_value in enumerate(all_dates)
    }

    target_fold_values = [
        date_to_fold.get(pd.Timestamp(d), 0)
        for d in target_index.normalize()
    ]

    pool_fold_values = [
        date_to_fold.get(pd.Timestamp(d), 0)
        for d in pool_index.normalize()
    ]

    target_folds = pd.Series(
        target_fold_values,
        index=target_index,
        dtype="int64",
    )

    pool_folds = pd.Series(
        pool_fold_values,
        index=pool_index,
        dtype="int64",
    )

    return target_folds, pool_folds


def _smooth_reference_within_seasons(
    series: pd.Series,
    halflife_steps: int,
) -> pd.Series:
    """
    V5 CHANGE:
    Smooth continuously inside each configured cooling season.

    The filter does not reset at midnight, but separate configured cooling seasons are
    kept independent so the long inter-season gap is never bridged.
    """
    result = series.copy()

    for _, (start, end) in SEASONS.items():
        start_ts = pd.Timestamp(start)
        end_ts = pd.Timestamp(end)

        mask = (
            (series.index >= start_ts)
            & (series.index <= end_ts)
        )

        season = series.loc[mask]

        if season.empty:
            continue

        result.loc[mask] = season.ewm(
            halflife=halflife_steps,
            adjust=False,
            min_periods=1,
        ).mean()

    return result


def estimate_active_reference(
    target: pd.DataFrame,
    pool: pd.DataFrame,
    chiller_col: str,
) -> pd.DataFrame:
    """
    Estimate the V5 Smooth Thermal Trend Reference (STTR).

    The output column names remain compatible with the rest of the V4
    notebook so the state, plotting, caching, and Excel workflow can remain
    unchanged.
    """
    result = pd.DataFrame(index=target.index)

    result["active_reference_raw_kw"] = np.nan
    result["active_reference_kw"] = np.nan
    result["active_reference_floor_kw"] = np.nan
    result["active_reference_ceiling_kw"] = np.nan
    result["active_reference_mad_kw"] = np.nan
    result["active_reference_level"] = ""
    result["active_reference_neighbors"] = 0  # Not applicable to STTR.
    result["active_reference_fold"] = np.nan

    pool = pool.dropna(
        subset=[chiller_col] + REFERENCE_FEATURES
    ).copy()

    if pool.empty:
        raise ValueError("STTR active pool is empty.")

    global_median = float(pool[chiller_col].median())
    global_mad = float(
        (pool[chiller_col] - global_median).abs().median()
    )
    global_floor = float(
        pool[chiller_col].quantile(STTR_FLOOR_Q)
    )
    global_ceiling = float(
        pool[chiller_col].quantile(STTR_CEILING_Q)
    )

    target_folds, pool_folds = _shared_calendar_day_folds(
        target.index,
        pool.index,
        STTR_N_SPLITS,
    )

    for fold in range(STTR_N_SPLITS):

        train = pool.loc[
            pool_folds != fold
        ].copy()

        fold_target_index = target.index[
            target_folds == fold
        ]

        complete_target = (
            target.loc[
                fold_target_index,
                REFERENCE_FEATURES,
            ]
            .notna()
            .all(axis=1)
        )

        predict_index = fold_target_index[
            complete_target
        ]

        # If a building/fold has too little active evidence, keep the run alive
        # with an explicit fallback rather than fitting an unstable spline.
        if len(train) < STTR_MIN_TRAIN_ACTIVE_SAMPLES:

            result.loc[
                fold_target_index,
                "active_reference_raw_kw",
            ] = global_median

            result.loc[
                fold_target_index,
                "active_reference_floor_kw",
            ] = global_floor

            result.loc[
                fold_target_index,
                "active_reference_ceiling_kw",
            ] = global_ceiling

            result.loc[
                fold_target_index,
                "active_reference_mad_kw",
            ] = global_mad

            result.loc[
                fold_target_index,
                "active_reference_level",
            ] = "sttr_insufficient_training_fallback"

            result.loc[
                fold_target_index,
                "active_reference_fold",
            ] = fold

            continue

        y_raw = train[chiller_col].to_numpy(float)

        y_low = float(
            np.quantile(y_raw, STTR_TARGET_LOW_Q)
        )
        y_high = float(
            np.quantile(y_raw, STTR_TARGET_HIGH_Q)
        )

        y_train = np.clip(
            y_raw,
            y_low,
            y_high,
        )

        fold_floor = float(
            np.quantile(y_raw, STTR_FLOOR_Q)
        )
        fold_ceiling = float(
            np.quantile(y_raw, STTR_CEILING_Q)
        )

        fold_median = float(
            np.median(y_raw)
        )
        fold_mad = float(
            np.median(
                np.abs(y_raw - fold_median)
            )
        )

        model = _make_sttr_model()

        model.fit(
            train[REFERENCE_FEATURES].to_numpy(float),
            y_train,
        )

        if len(predict_index):

            prediction = model.predict(
                target.loc[
                    predict_index,
                    REFERENCE_FEATURES,
                ].to_numpy(float)
            )

            prediction = np.clip(
                prediction,
                fold_floor,
                fold_ceiling,
            )

            result.loc[
                predict_index,
                "active_reference_raw_kw",
            ] = prediction

            result.loc[
                predict_index,
                "active_reference_floor_kw",
            ] = fold_floor

            result.loc[
                predict_index,
                "active_reference_ceiling_kw",
            ] = fold_ceiling

            result.loc[
                predict_index,
                "active_reference_mad_kw",
            ] = fold_mad

            result.loc[
                predict_index,
                "active_reference_level",
            ] = "sttr_crossfit"

            result.loc[
                predict_index,
                "active_reference_fold",
            ] = fold

        del model
        gc.collect()

    # Weather gaps or any unresolved timestamps keep a transparent fallback.
    unresolved = result["active_reference_raw_kw"].isna()

    result.loc[
        unresolved,
        "active_reference_raw_kw",
    ] = global_median

    result.loc[
        unresolved,
        "active_reference_floor_kw",
    ] = global_floor

    result.loc[
        unresolved,
        "active_reference_ceiling_kw",
    ] = global_ceiling

    result.loc[
        unresolved,
        "active_reference_mad_kw",
    ] = global_mad

    result.loc[
        unresolved,
        "active_reference_level",
    ] = "global_active_median"

    # V5 CHANGE — continuous causal smoothing within each cooling season.
    result["active_reference_kw"] = (
        _smooth_reference_within_seasons(
            result["active_reference_raw_kw"],
            STTR_SMOOTH_HALFLIFE_STEPS,
        )
    )

    # Re-apply the physical expected-active range after smoothing.
    result["active_reference_kw"] = np.minimum(
        np.maximum(
            result["active_reference_kw"],
            result["active_reference_floor_kw"],
        ),
        result["active_reference_ceiling_kw"],
    )

    return result

In [ ]:
# CELL 4 — State assignment and night summary

STATE_ORDER = ["OFF", "AUX", "ACTIVE-PART", "ACTIVE-HIGH", "AMBIGUOUS"]
STATE_COLORS = {
    # OFF/AUX remain visually distinct from cooling operation.
    # PART and ACTIVE deliberately use the same blue family because both
    # represent Chiller ON; PART is simply shown with a lighter shade.
    "OFF": "#c7c7c7",
    "AUX": "#f6d77a",
    "ACTIVE-PART": "#a9d6f5",
    "ACTIVE-HIGH": "#4a90d9",
    "AMBIGUOUS": "#b9a0d8",
}


def _ambiguous_near_boundary(ratio: float, boundaries: List[float], margin: float) -> bool:
    return any(abs(ratio - b) <= margin for b in boundaries)


def classify_reference_states(
    df: pd.DataFrame,
    cfg: dict,
    chiller_col: str,
) -> pd.DataFrame:
    """
    Assign OFF / AUX / ACTIVE-PART / ACTIVE-HIGH.

    Plan A uses the weather-adjusted percentage.
    Plan B keeps the previous fixed-kW logic as a fallback only.
    """
    out = df.copy()
    load = out[chiller_col].astype(float)

    if cfg["reference_method"].lower() == "fixed_threshold":
        threshold = float(cfg["legacy_fixed_threshold_kw"])
        out["active_ratio"] = np.nan
        out["reference_state_raw"] = np.where(
            load.isna(), np.nan, np.where(load > threshold, "ACTIVE-HIGH", "OFF")
        )
    else:
        ref = out["active_reference_kw"].replace(0, np.nan)
        ratio = load / ref
        out["active_ratio"] = ratio

        states = pd.Series(index=out.index, dtype="object")
        valid = load.notna() & ratio.notna()

        states.loc[valid & (ratio <= cfg["off_max_ratio"])] = "OFF"
        states.loc[
            valid
            & (ratio > cfg["off_max_ratio"])
            & (ratio <= cfg["aux_max_ratio"])
        ] = "AUX"
        states.loc[
            valid
            & (ratio > cfg["aux_max_ratio"])
            & (ratio <= cfg["part_max_ratio"])
        ] = "ACTIVE-PART"
        states.loc[valid & (ratio > cfg["part_max_ratio"])] = "ACTIVE-HIGH"

        # Optional absolute near-zero override. It is OFF by default because the
        # main strategy is percentage-based.
        if cfg.get("use_absolute_near_zero", False):
            states.loc[load.notna() & (load <= cfg["near_zero_max_kw"])] = "OFF"

        # Optional uncertainty band. The code is ready for later use but all
        # buildings start with this option disabled.
        if cfg.get("use_ambiguous", False):
            boundaries = [
                cfg["off_max_ratio"],
                cfg["aux_max_ratio"],
                cfg["part_max_ratio"],
            ]
            margin = cfg["ambiguous_margin"]
            for ts in states.index[valid]:
                r = float(ratio.at[ts])
                if _ambiguous_near_boundary(r, boundaries, margin):
                    states.at[ts] = "AMBIGUOUS"

        out["reference_state_raw"] = states

    out["reference_state"] = smooth_short_state_runs(
        out["reference_state_raw"],
        min_steps=STATE_MIN_PERSISTENCE_STEPS,
    )
    return out


def smooth_short_state_runs(states: pd.Series, min_steps: int = 2) -> pd.Series:
    """
    Remove only isolated short flips when the state before and after is the same.
    Genuine transitions are left unchanged.
    """
    s = states.copy()
    valid = s.notna()
    if not valid.any() or min_steps <= 1:
        return s

    arr = s.to_numpy(object)
    n = len(arr)
    i = 0
    while i < n:
        if pd.isna(arr[i]):
            i += 1
            continue
        j = i + 1
        while j < n and arr[j] == arr[i]:
            j += 1

        run_len = j - i
        if run_len < min_steps and i > 0 and j < n:
            left = arr[i - 1]
            right = arr[j]
            if pd.notna(left) and left == right:
                arr[i:j] = left
        i = j

    return pd.Series(arr, index=s.index, name=s.name)

In [ ]:
# CELL 5 — Load Actual building data + weather and resolve source cooling seasons

def load_weather_for_nn(path: Path):
    raw = pd.read_excel(path, sheet_name=CFG.weather_sheet)
    if WEATHER_DATETIME_COL not in raw.columns:
        raise KeyError(f'Weather file must contain {WEATHER_DATETIME_COL}')
    if TEMPERATURE_COL not in raw.columns:
        raise KeyError(f'Weather file must contain {TEMPERATURE_COL}')

    direct_ok = DIRECT_RADIATION_COL in raw.columns
    short_ok = SHORTWAVE_RADIATION_COL in raw.columns
    if not direct_ok and not short_ok:
        raise KeyError('Weather file requires direct_radiation or shortwave_radiation.')

    direct_complete = (
        pd.to_numeric(raw[DIRECT_RADIATION_COL], errors='coerce').notna().mean()
        if direct_ok else 0.0
    )
    if direct_ok and direct_complete >= DIRECT_RADIATION_COMPLETENESS_THRESHOLD:
        radiation_col = DIRECT_RADIATION_COL
    elif short_ok:
        radiation_col = SHORTWAVE_RADIATION_COL
    else:
        radiation_col = DIRECT_RADIATION_COL

    w = raw[[WEATHER_DATETIME_COL, TEMPERATURE_COL, radiation_col]].copy()
    w[WEATHER_DATETIME_COL] = pd.to_datetime(w[WEATHER_DATETIME_COL], errors='coerce')
    w[TEMPERATURE_COL] = pd.to_numeric(w[TEMPERATURE_COL], errors='coerce')
    w[radiation_col] = pd.to_numeric(w[radiation_col], errors='coerce')
    w = (w.dropna(subset=[WEATHER_DATETIME_COL])
           .sort_values(WEATHER_DATETIME_COL)
           .drop_duplicates(WEATHER_DATETIME_COL, keep='last')
           .set_index(WEATHER_DATETIME_COL))
    full_index = pd.date_range(w.index.min(), w.index.max(), freq=f'{FREQ_MIN}min')
    w = w.reindex(full_index)
    w.index.name = DATETIME_COL
    return w, radiation_col, {
        'direct_radiation_completeness': float(direct_complete),
        'selected_radiation_column': radiation_col,
    }

if not LOAD_FILE.exists():
    raise FileNotFoundError(f'Building-load file not found: {LOAD_FILE}')
if not WEATHER_FILE.exists():
    raise FileNotFoundError(f'Weather file not found: {WEATHER_FILE}')

load_raw = pd.read_excel(LOAD_FILE, sheet_name=CFG.load_sheet)
if DATETIME_COL not in load_raw.columns:
    raise KeyError(f'{LOAD_FILE.name} must contain {DATETIME_COL}')
load_raw[DATETIME_COL] = pd.to_datetime(load_raw[DATETIME_COL], errors='coerce')
load_raw = (load_raw.dropna(subset=[DATETIME_COL])
                    .sort_values(DATETIME_COL)
                    .drop_duplicates(DATETIME_COL, keep='last')
                    .set_index(DATETIME_COL))

# Exact 15-minute grid. Inserted timestamps remain NaN here; interpolation is done later
# only for the NN inputs, never for the authoritative chiller truth generation.
load_index = pd.date_range(load_raw.index.min(), load_raw.index.max(), freq=f'{FREQ_MIN}min')
inserted_load_timestamps = load_index.difference(load_raw.index)
load_raw = load_raw.reindex(load_index)
load_raw.index.name = DATETIME_COL

for b, c in BUILDINGS.items():
    for col in [c['total'], c['chiller']]:
        if col not in load_raw.columns:
            raise KeyError(f'Missing required building column: {col}')
        load_raw[col] = pd.to_numeric(load_raw[col], errors='coerce')

weather_raw, RADIATION_COL, WEATHER_QC = load_weather_for_nn(WEATHER_FILE)
weather_features = add_weather_features(weather_raw, RADIATION_COL)

# Resolve source cooling seasons from a private season plan.
# The original availability-based clipping is retained; only the original
# date literals are read from a local, unpublished configuration.
def _last_valid_timestamp(series):
    s = pd.to_numeric(series, errors='coerce')
    return pd.Timestamp(s.index[s.notna()].max()) if s.notna().any() else pd.NaT

last_valids = []
for b, c in BUILDINGS.items():
    last_valids += [_last_valid_timestamp(load_raw[c['total']]), _last_valid_timestamp(load_raw[c['chiller']])]
last_valids += [_last_valid_timestamp(weather_raw[TEMPERATURE_COL]), _last_valid_timestamp(weather_raw[RADIATION_COL])]
last_valids = [x for x in last_valids if pd.notna(x)]
if not last_valids:
    raise RuntimeError('Could not resolve common data availability.')
latest_joint_available = min(last_valids)

SEASONS.clear()
_season_file = TASK_DIR / 'data/season_windows.json'
if not _season_file.exists():
    raise FileNotFoundError(f'Missing cooling-season configuration: {_season_file}')
_season_windows = json.loads(_season_file.read_text(encoding='utf-8'))
for _season_name, _spec in _season_windows.items():
    _start = pd.Timestamp(_spec['start'])
    _end = pd.Timestamp(_spec['end'])
    _year = _start.year
    if _year not in set(load_raw.index.year) or _year not in set(weather_raw.index.year):
        continue
    if bool(_spec.get('clip_to_joint_availability', False)):
        _end = min(_end, latest_joint_available)
    if _end >= _start:
        SEASONS[_season_name] = (_start, _end)
if not SEASONS:
    raise RuntimeError('No configured cooling season intersects both input datasets.')

print('Building file:', LOAD_FILE)
print('Inserted missing building timestamps:', len(inserted_load_timestamps))
print('Weather QC:', WEATHER_QC)
print('Resolved seasons:')
for k, v in SEASONS.items():
    print(' ', k, v)

In [ ]:
# CELL 6 — Build/cache authoritative binary truth only
STATE_TO_BINARY = {
    'OFF': 0.0,
    'AUX': 0.0,
    'ACTIVE-PART': 1.0,
    'ACTIVE-HIGH': 1.0,
}

def season_mask(index):
    mask = np.zeros(len(index), dtype=bool)
    for start, end in SEASONS.values():
        mask |= (index >= start) & (index <= end)
    return mask

def prepare_actual_building_source_frame(building):
    cfg = BUILDINGS[building]
    base = load_raw[[cfg['total'], cfg['chiller']]].copy()
    has_data = base.notna().any(axis=1)
    if not has_data.any():
        raise ValueError(f'{building}: no Actual Total/Chiller data found.')
    base = base.loc[has_data[has_data].index[0]:has_data[has_data].index[-1]]
    base = base.reindex(pd.date_range(base.index.min(), base.index.max(), freq=f'{FREQ_MIN}min'))
    base.index.name = DATETIME_COL
    working = base.join(
        weather_features[[TEMPERATURE_COL, RADIATION_COL] + REFERENCE_FEATURES],
        how='left',
    )
    return working.loc[season_mask(working.index)].copy()

def build_one_building_truth(building):
    cfg = dict(BUILDINGS[building])
    cache_path = TRUTH_DIR / f'{building}_truth.pkl.gz'
    summary_path = TRUTH_DIR / f'{building}_truth_summary.json'

    if (
        not CFG.rebuild_truth
        and cache_path.exists()
        and summary_path.exists()
    ):
        work = pd.read_pickle(cache_path, compression='gzip')
        if 'binary_on' not in work.columns:
            work['binary_on'] = work['reference_state'].map(STATE_TO_BINARY)
        with open(summary_path, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        return {'work': work, 'summary': summary}

    working = prepare_actual_building_source_frame(building)

    active_pool, seed_meta = build_active_seed_pool(
        df=working,
        chiller_col=cfg['chiller'],
        weather_features=weather_features,
        seed_quantile=cfg['active_seed_quantile'],
        min_persistence_steps=ACTIVE_SEED_MIN_PERSISTENCE_STEPS,
    )

    active_ref = estimate_active_reference(
        target=working,
        pool=active_pool,
        chiller_col=cfg['chiller'],
    )

    for col in active_ref.columns:
        working[col] = active_ref[col]

    working = classify_reference_states(
        df=working,
        cfg=cfg,
        chiller_col=cfg['chiller'],
    )
    working['binary_on'] = working['reference_state'].map(STATE_TO_BINARY)

    valid = working['binary_on'].notna()
    summary = {
        'building': building,
        'valid_binary_intervals': int(valid.sum()),
        'binary_coverage': float(valid.mean()),
        'on_fraction': float(working.loc[valid, 'binary_on'].mean()) if valid.any() else np.nan,
        'active_seed_meta': seed_meta,
    }

    working.to_pickle(cache_path, compression='gzip')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, default=str)

    return {'work': working, 'summary': summary}

TRUTH = {}
for b in BUILDING_NAMES:
    print('Authoritative binary truth:', b)
    TRUTH[b] = build_one_building_truth(b)

truth_summary = pd.DataFrame([TRUTH[b]['summary'] for b in BUILDING_NAMES])
truth_summary.to_csv(TRUTH_DIR / 'authoritative_binary_truth_summary.csv', index=False)
display(truth_summary)

for b in BUILDING_NAMES:
    values = set(pd.Series(TRUTH[b]['work']['binary_on']).dropna().unique().tolist())
    assert values.issubset({0.0, 1.0}), (b, values)

print('Binary truth validated for all buildings.')

In [ ]:
# CELL 7 — Shared operational-event semantics
# Used for training metadata and held-out evaluation.

def regularize_binary_state(state):
    s = pd.to_numeric(state, errors='coerce').astype(float).sort_index()
    if s.empty:
        return s
    s = s.loc[~s.index.duplicated(keep='last')]
    full_idx = pd.date_range(s.index.min(), s.index.max(), freq=f'{CFG.dt_minutes}min')
    return s.reindex(full_idx)

def vectorized_transition_masks(state):
    s = regularize_binary_state(state)
    if s.empty:
        empty = pd.Series(False, index=s.index, dtype=bool)
        return s, empty, empty.copy(), empty.copy()

    pre_steps = int(CFG.event_pre_window_min / CFG.dt_minutes)
    min_valid = int(math.ceil(pre_steps * CFG.event_pre_min_valid_fraction))
    lagged = s.shift(1)

    pre_valid_n = lagged.notna().astype(np.int8).rolling(pre_steps, min_periods=1).sum()
    pre_off_n = lagged.eq(0).astype(np.int8).rolling(pre_steps, min_periods=1).sum()
    pre_on_n = lagged.eq(1).astype(np.int8).rolling(pre_steps, min_periods=1).sum()
    denom = pre_valid_n.replace(0, np.nan)

    pre_off_ratio = pre_off_n / denom
    pre_on_ratio = pre_on_n / denom
    prev = s.shift(1)

    post_on = pd.Series(True, index=s.index, dtype=bool)
    post_off = pd.Series(True, index=s.index, dtype=bool)
    for j in range(CFG.event_post_steps):
        future = s.shift(-j)
        post_on &= future.eq(1)
        post_off &= future.eq(0)

    enough_pre = pre_valid_n >= min_valid

    valid_on = (
        enough_pre
        & (pre_off_ratio >= CFG.event_pre_dominance_ratio)
        & prev.eq(0)
        & post_on
    ).fillna(False)

    valid_off = (
        enough_pre
        & (pre_on_ratio >= CFG.event_pre_dominance_ratio)
        & prev.eq(1)
        & post_off
    ).fillna(False)

    raw_transition = (
        s.notna()
        & prev.notna()
        & s.ne(prev)
    ).fillna(False)

    return s, valid_on.astype(bool), valid_off.astype(bool), raw_transition.astype(bool)

def _sustained_state_at(regular_state, ts, required_state):
    ts = pd.Timestamp(ts)
    dt = pd.Timedelta(minutes=CFG.dt_minutes)
    pre_steps = int(CFG.event_pre_window_min / CFG.dt_minutes)

    pre_idx = pd.date_range(
        ts - pre_steps * dt,
        ts - dt,
        freq=f'{CFG.dt_minutes}min',
    )
    pre = regular_state.reindex(pre_idx)
    n_valid = int(pre.notna().sum())
    min_valid = int(math.ceil(pre_steps * CFG.event_pre_min_valid_fraction))
    if n_valid < min_valid:
        return False

    dominance = float((pre.dropna() == required_state).mean())
    if dominance < CFG.event_pre_dominance_ratio:
        return False

    for j in range(CFG.event_post_steps):
        tj = ts + pd.Timedelta(minutes=CFG.dt_minutes * j)
        if (
            tj not in regular_state.index
            or pd.isna(regular_state.loc[tj])
            or int(regular_state.loc[tj]) != required_state
        ):
            return False
    return True

def build_daily_operational_table(state):
    s, valid_on_mask, valid_off_mask, raw_transition_mask = vectorized_transition_masks(state)
    if s.empty:
        return pd.DataFrame()

    on_times = valid_on_mask.index[valid_on_mask.to_numpy()]
    off_times = valid_off_mask.index[valid_off_mask.to_numpy()]
    raw_times = raw_transition_mask.index[raw_transition_mask.to_numpy()]

    on_by_day, off_by_day, raw_by_day = {}, {}, {}
    for ts in on_times:
        on_by_day.setdefault(ts.normalize(), []).append(ts)
    for ts in off_times:
        off_by_day.setdefault(ts.normalize(), []).append(ts)
    for ts in raw_times:
        raw_by_day.setdefault(ts.normalize(), []).append(ts)

    rows = []
    first_day = s.index.min().normalize()
    last_day = s.index.max().normalize()

    for date in pd.date_range(first_day, last_day, freq='D'):
        day_end = date + pd.Timedelta(hours=23, minutes=45)
        day_vals = s.loc[date:day_end]
        day_coverage = float(day_vals.notna().mean())
        valid_vals = day_vals.dropna()

        always_on = bool(
            day_coverage >= CFG.day_status_min_coverage
            and len(valid_vals)
            and valid_vals.eq(1).all()
        )
        always_off = bool(
            day_coverage >= CFG.day_status_min_coverage
            and len(valid_vals)
            and valid_vals.eq(0).all()
        )

        search_start = pd.Timestamp(f'{date.date()} {CFG.morning_start}')
        search_end = pd.Timestamp(f'{date.date()} {CFG.morning_end}')

        day_on = on_by_day.get(date, [])
        day_off = off_by_day.get(date, [])

        morning_on = [ts for ts in day_on if search_start <= ts <= search_end]
        on_transition = morning_on[0] if morning_on else pd.NaT

        # Retained left-edge reporting for start time.
        on_time = (
            on_transition - pd.Timedelta(minutes=CFG.dt_minutes)
            if pd.notna(on_transition)
            else pd.NaT
        )

        already_on_at_04 = _sustained_state_at(s, search_start, 1)

        # Principal shutdown semantics:
        #   * if a principal morning start exists, use the last validated OFF after it;
        #   * if the chiller was already ON at 04:00, use the last validated OFF after 04:00;
        #   * otherwise DO NOT assign a later-cycle shutdown as the principal daily shutdown.
        if pd.notna(on_transition):
            eligible_off = [ts for ts in day_off if ts > on_transition]
        elif already_on_at_04:
            eligible_off = [ts for ts in day_off if ts >= search_start]
        else:
            eligible_off = []

        off_time = eligible_off[-1] if eligible_off else pd.NaT

        if day_coverage < CFG.day_status_min_coverage:
            day_status = 'INSUFFICIENT_DATA'
        elif always_on:
            day_status = 'ALWAYS_ON'
        elif always_off:
            day_status = 'ALWAYS_OFF'
        elif pd.notna(on_time) and pd.notna(off_time):
            day_status = 'START_AND_STOP'
        elif pd.notna(on_time):
            day_status = 'START_ONLY'
        elif pd.notna(off_time):
            day_status = 'STOP_ONLY'
        elif already_on_at_04:
            day_status = 'ALREADY_ON_NO_VALID_STOP'
        else:
            day_status = 'MIXED_NO_PRIMARY_EVENT'

        rows.append({
            'date': date,
            'turn_on_transition_time': on_transition,
            'turn_on_time': on_time,
            'turn_off_time': off_time,
            'already_on_at_04': bool(already_on_at_04),
            'always_on': always_on,
            'always_off': always_off,
            'day_status': day_status,
            'day_state_coverage': day_coverage,
            'n_valid_on_candidates': int(len(day_on)),
            'n_valid_off_candidates': int(len(day_off)),
            'n_raw_transitions': int(len(raw_by_day.get(date, []))),
        })

    return pd.DataFrame(rows)

def event_distance_from_mask(index, event_mask):
    mask = pd.Series(event_mask, index=index).fillna(False).astype(bool).to_numpy()
    n = len(mask)
    inf = 10**9
    dist = np.full(n, inf, dtype=np.int64)

    last = -inf
    for i in range(n):
        if mask[i]:
            last = i
        if last > -inf:
            dist[i] = i - last

    last = inf
    for i in range(n - 1, -1, -1):
        if mask[i]:
            last = i
        if last < inf:
            dist[i] = min(dist[i], last - i)

    return pd.Series(np.minimum(dist, 99), index=index, dtype=float)

def build_training_event_metadata(binary_state):
    s = regularize_binary_state(binary_state)
    daily = build_daily_operational_table(s)

    primary = pd.Series(False, index=s.index, dtype=bool)
    primary_type = pd.Series('', index=s.index, dtype=object)

    if not daily.empty:
        for row in daily.itertuples(index=False):
            on_ts = row.turn_on_transition_time
            off_ts = row.turn_off_time

            if pd.notna(on_ts) and pd.Timestamp(on_ts) in primary.index:
                primary.loc[pd.Timestamp(on_ts)] = True
                primary_type.loc[pd.Timestamp(on_ts)] = 'PRIMARY_ON'

            if pd.notna(off_ts) and pd.Timestamp(off_ts) in primary.index:
                primary.loc[pd.Timestamp(off_ts)] = True
                primary_type.loc[pd.Timestamp(off_ts)] = 'PRIMARY_OFF'

    prev = s.shift(1)
    all_transition = (s.notna() & prev.notna() & s.ne(prev)).fillna(False)
    secondary = all_transition & ~primary

    return pd.DataFrame({
        'primary_event_exact': primary.astype(float),
        'primary_event_type': primary_type,
        'secondary_transition_exact': secondary.astype(float),
        'primary_event_distance': event_distance_from_mask(s.index, primary),
        'secondary_transition_distance': event_distance_from_mask(s.index, secondary),
    }, index=s.index)

TRAINING_EVENT_META = {}
for b in BUILDING_NAMES:
    TRAINING_EVENT_META[b] = build_training_event_metadata(TRUTH[b]['work']['binary_on'])
    m = TRAINING_EVENT_META[b]
    print(
        f'{b}: primary={int(m["primary_event_exact"].sum())}, '
        f'secondary={int(m["secondary_transition_exact"].sum())}'
    )

In [ ]:
# CELL 8 — Safe interpolation, H/V/G features, and event-aware training metadata

def interpolate_only_short_runs(series: pd.Series, max_gap_steps: int) -> pd.Series:
    """Interpolate only internal NaN runs whose complete run length <= max_gap_steps."""
    s = pd.to_numeric(series, errors='coerce').astype(float).copy()
    na = s.isna()
    if not na.any():
        return s
    run_id = (na != na.shift(fill_value=False)).cumsum()
    run_len = na.groupby(run_id).transform('sum')
    candidate = s.interpolate(method='linear', limit_direction='both', limit_area='inside')
    fillable = na & (run_len <= int(max_gap_steps))
    s.loc[fillable] = candidate.loc[fillable]
    return s


def empirical_percentile(values: pd.Series) -> pd.Series:
    x = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=x.index, dtype=float)
    m = x.notna()
    n = int(m.sum())
    if n == 0:
        return out
    if n == 1:
        out.loc[m] = 0.5
        return out
    ranks = x.loc[m].rank(method='average')
    out.loc[m] = (ranks - 1.0) / (n - 1.0)
    return out


def robust_median_iqr(values):
    x = pd.to_numeric(pd.Series(values), errors='coerce').dropna().to_numpy(float)
    if len(x) == 0:
        return np.nan, np.nan
    med = float(np.median(x))
    q25, q75 = np.quantile(x, [0.25, 0.75])
    return med, float(q75 - q25)


def robust_z(x, med, iqr):
    x = np.asarray(x, float)
    med = np.asarray(med, float)
    iqr = np.asarray(iqr, float)
    denom = iqr / 1.349
    z = np.zeros(np.broadcast(x, med, iqr).shape, dtype=float)
    valid = np.isfinite(x) & np.isfinite(med)
    strong = valid & np.isfinite(denom) & (np.abs(denom) > 1e-9)
    z[strong] = (x[strong] - med[strong]) / denom[strong]
    z[valid & ~strong] = 0.0
    z[~valid] = np.nan
    return np.clip(z, -CFG.robust_z_clip, CFG.robust_z_clip)


def safe_ratio(num, den, default=0.0):
    num = np.asarray(num, float)
    den = np.asarray(den, float)
    out = np.full(np.broadcast(num, den).shape, default, dtype=float)
    good = np.isfinite(num) & np.isfinite(den) & (np.abs(den) > 1e-9)
    out[good] = num[good] / den[good]
    return np.clip(out, CFG.ratio_clip_low, CFG.ratio_clip_high)


def add_context_statistics(df: pd.DataFrame, value_col: str, prefix: str, load_mode: bool) -> pd.DataFrame:
    """
    Add H/V/G percentile + robust distribution descriptors for one variable.

    v2 QC:
      - Horizontal context is valid only when >=75% of the 96 daily grid points
        are available for this derived variable.
      - Vertical context is valid only when the same clock slot has observations
        on at least CFG.min_vertical_context_days valid days.
      - No seasonal or neighboring-slot fallback is fabricated.
    """
    out = df.copy()
    x = pd.to_numeric(out[value_col], errors='coerce').astype(float)

    # Global month descriptors.
    g_med, g_iqr = robust_median_iqr(x)
    out[f'{prefix}_g_score'] = empirical_percentile(x)
    out[f'{prefix}_g_med'] = g_med
    out[f'{prefix}_g_iqr'] = g_iqr
    out[f'{prefix}_g_z'] = robust_z(
        x.to_numpy(),
        np.full(len(out), g_med),
        np.full(len(out), g_iqr),
    )

    # Horizontal: same day.
    out['_date'] = out.index.normalize()
    h_score = pd.Series(np.nan, index=out.index, dtype=float)
    h_med = pd.Series(np.nan, index=out.index, dtype=float)
    h_iqr = pd.Series(np.nan, index=out.index, dtype=float)
    min_h = int(math.ceil(CFG.steps_per_day * CFG.min_horizontal_context_fraction))

    for _, idx in out.groupby('_date').groups.items():
        vals = x.loc[idx]
        if int(vals.notna().sum()) < min_h:
            continue
        med, iqr = robust_median_iqr(vals)
        h_score.loc[idx] = empirical_percentile(vals)
        h_med.loc[idx] = med
        h_iqr.loc[idx] = iqr

    out[f'{prefix}_h_score'] = h_score
    out[f'{prefix}_h_med'] = h_med
    out[f'{prefix}_h_iqr'] = h_iqr
    out[f'{prefix}_h_z'] = robust_z(
        x.to_numpy(),
        h_med.to_numpy(),
        h_iqr.to_numpy(),
    )

    # Vertical: same 15-min clock position across days of the calendar month.
    out['_slot'] = (
        (out.index.hour * 60 + out.index.minute) // CFG.dt_minutes
    ).astype(int)

    v_score = pd.Series(np.nan, index=out.index, dtype=float)
    v_med = pd.Series(np.nan, index=out.index, dtype=float)
    v_iqr = pd.Series(np.nan, index=out.index, dtype=float)

    for _, idx in out.groupby('_slot').groups.items():
        vals = x.loc[idx]
        if int(vals.notna().sum()) < CFG.min_vertical_context_days:
            continue
        med, iqr = robust_median_iqr(vals)
        v_score.loc[idx] = empirical_percentile(vals)
        v_med.loc[idx] = med
        v_iqr.loc[idx] = iqr

    out[f'{prefix}_v_score'] = v_score
    out[f'{prefix}_v_med'] = v_med
    out[f'{prefix}_v_iqr'] = v_iqr
    out[f'{prefix}_v_z'] = robust_z(
        x.to_numpy(),
        v_med.to_numpy(),
        v_iqr.to_numpy(),
    )

    if load_mode:
        # Keep building-load scale dimensionless for cross-building transfer.
        out[f'{prefix}_h_med_rel'] = safe_ratio(
            h_med.to_numpy(), np.full(len(out), g_med)
        )
        out[f'{prefix}_h_iqr_rel'] = safe_ratio(
            h_iqr.to_numpy(), np.full(len(out), g_iqr)
        )
        out[f'{prefix}_v_med_rel'] = safe_ratio(
            v_med.to_numpy(), np.full(len(out), g_med)
        )
        out[f'{prefix}_v_iqr_rel'] = safe_ratio(
            v_iqr.to_numpy(), np.full(len(out), g_iqr)
        )
        out[f'{prefix}_g_riqr'] = safe_ratio(
            np.full(len(out), g_iqr), np.full(len(out), abs(g_med))
        )

    return out


LOAD_FEATURES = [
    'p_h_score','p_h_z','p_h_med_rel','p_h_iqr_rel',
    'p_v_score','p_v_z','p_v_med_rel','p_v_iqr_rel',
    'p_g_score','p_g_z','p_g_riqr',
]
TEMP_FEATURES = [
    't_h_score','t_h_z','t_h_med','t_h_iqr',
    't_v_score','t_v_z','t_v_med','t_v_iqr',
    't_g_score','t_g_z','t_g_med','t_g_iqr',
]
RAD_FEATURES = [
    'r_h_score','r_h_z','r_h_med','r_h_iqr',
    'r_v_score','r_v_z','r_v_med','r_v_iqr',
    'r_g_score','r_g_z','r_g_med','r_g_iqr',
]
DELTA_FEATURES = ['p_delta15','p_delta30','p_delta60']
CALENDAR_FEATURES = ['tod_sin','tod_cos','is_weekend']
FEATURE_NAMES = LOAD_FEATURES + TEMP_FEATURES + RAD_FEATURES + DELTA_FEATURES + CALENDAR_FEATURES
assert len(FEATURE_NAMES) == 41, len(FEATURE_NAMES)


# Interpolate weather only across short gaps, then make exactly ONE causal 4-h aggregate per timestamp.
# Keep the trained v1 semantics exactly: all 16 preceding 15-min values are required.
_temp_interp = interpolate_only_short_runs(weather_raw[TEMPERATURE_COL], CFG.weather_max_interp_gap_steps)
_rad_interp = interpolate_only_short_runs(weather_raw[RADIATION_COL], CFG.weather_max_interp_gap_steps)
EFFECTIVE_WEATHER = pd.DataFrame(index=weather_raw.index)
EFFECTIVE_WEATHER['temp_4h'] = _temp_interp.shift(1).rolling(
    CFG.weather_backward_steps, min_periods=CFG.weather_backward_min_periods
).mean()
EFFECTIVE_WEATHER['rad_4h'] = _rad_interp.shift(1).rolling(
    CFG.weather_backward_steps, min_periods=CFG.weather_backward_min_periods
).mean()

# Original weather coverage is evaluated before interpolation.
WEATHER_DAY_COVERAGE = pd.DataFrame(index=pd.Index(sorted(set(weather_raw.index.normalize())), name='date'))
for c, name in [(TEMPERATURE_COL,'temp_cov'), (RADIATION_COL,'rad_cov')]:
    s = pd.to_numeric(weather_raw[c], errors='coerce')
    WEATHER_DAY_COVERAGE[name] = s.notna().groupby(s.index.normalize()).mean().reindex(WEATHER_DAY_COVERAGE.index)


def build_building_feature_table(building: str) -> pd.DataFrame:
    cache_path = FEATURE_DIR / f'{building}_context_features.pkl.gz'
    if cache_path.exists() and not CFG.rebuild_features:
        df = pd.read_pickle(cache_path, compression='gzip')
        required_meta = [
            'primary_event_distance',
            'secondary_transition_distance',
            'primary_event_exact',
            'secondary_transition_exact',
        ]
        missing = [
            c for c in FEATURE_NAMES + required_meta
            if c not in df.columns
        ]
        if not missing:
            return df

    cfg = BUILDINGS[building]
    # Full input stream is used so load deltas and 4-h weather can cross day/month boundaries.
    p_raw = pd.to_numeric(load_raw[cfg['total']], errors='coerce').astype(float)
    p = interpolate_only_short_runs(p_raw, CFG.ptotal_max_interp_gap_steps)

    base = pd.DataFrame(index=load_raw.index)
    base['ptotal'] = p
    base['temp4h'] = EFFECTIVE_WEATHER['temp_4h'].reindex(base.index)
    base['rad4h'] = EFFECTIVE_WEATHER['rad_4h'].reindex(base.index)
    base['true_state'] = TRUTH[building]['work']['binary_on'].reindex(base.index)

    # Daily original-input coverage; never count interpolated values as observed coverage.
    p_cov = p_raw.notna().groupby(p_raw.index.normalize()).mean()
    date_idx = base.index.normalize()
    base['ptotal_day_cov'] = pd.Series(date_idx, index=base.index).map(p_cov).to_numpy(float)
    base['temp_day_cov'] = pd.Series(date_idx, index=base.index).map(WEATHER_DAY_COVERAGE['temp_cov']).to_numpy(float)
    base['rad_day_cov'] = pd.Series(date_idx, index=base.index).map(WEATHER_DAY_COVERAGE['rad_cov']).to_numpy(float)

    # Restrict outputs to the source cooling seasons only.
    base = base.loc[season_mask(base.index)].copy()
    base['year'] = base.index.year
    base['month'] = base.index.month
    base['month_id'] = [f'{building}__{y:04d}-{m:02d}' for y,m in zip(base['year'],base['month'])]

    # Continuous load changes. Normalize later by within-day IQR (month-global fallback).
    p_full = p.reindex(base.index)
    raw_delta = {
        'p_delta15_raw': p_full - p.shift(1).reindex(base.index),
        'p_delta30_raw': p_full - p.shift(2).reindex(base.index),
        'p_delta60_raw': p_full - p.shift(4).reindex(base.index),
    }
    for k,v in raw_delta.items():
        base[k] = v

    month_parts = []
    for (year, month), mdf0 in base.groupby(['year','month']):
        mdf = mdf0.copy()
        # Only sufficiently observed days contribute to contextual distributions.
        eligible = (
            (mdf['ptotal_day_cov'] >= CFG.min_original_day_ptotal_coverage) &
            (mdf['temp_day_cov'] >= CFG.min_original_day_weather_coverage) &
            (mdf['rad_day_cov'] >= CFG.min_original_day_weather_coverage)
        )
        eligible_days = int(pd.Index(mdf.index[eligible].normalize()).nunique())
        if eligible_days < CFG.min_context_days_per_month:
            continue
        mdf = mdf.loc[eligible].copy()
        if mdf.empty:
            continue

        mdf = add_context_statistics(mdf, 'ptotal', 'p', load_mode=True)
        mdf = add_context_statistics(mdf, 'temp4h', 't', load_mode=False)
        mdf = add_context_statistics(mdf, 'rad4h', 'r', load_mode=False)

        # Normalize load deltas by the day's Ptotal IQR, with global-IQR fallback.
        day_iqr = pd.to_numeric(mdf['p_h_iqr'], errors='coerce').to_numpy(float)
        g_iqr = pd.to_numeric(mdf['p_g_iqr'], errors='coerce').to_numpy(float)
        denom = np.where(np.isfinite(day_iqr) & (np.abs(day_iqr)>1e-9), day_iqr, g_iqr)
        denom = np.where(np.isfinite(denom) & (np.abs(denom)>1e-9), denom, np.nan)
        for raw_name, out_name in [
            ('p_delta15_raw','p_delta15'),
            ('p_delta30_raw','p_delta30'),
            ('p_delta60_raw','p_delta60'),
        ]:
            d = pd.to_numeric(mdf[raw_name], errors='coerce').to_numpy(float) / denom
            mdf[out_name] = np.clip(d, -CFG.delta_clip, CFG.delta_clip)

        minutes = mdf.index.hour*60 + mdf.index.minute
        angle = 2*np.pi*minutes/(24*60)
        mdf['tod_sin'] = np.sin(angle)
        mdf['tod_cos'] = np.cos(angle)
        mdf['is_weekend'] = (mdf.index.dayofweek >= 5).astype(float)
        mdf['context_days'] = eligible_days

        month_parts.append(mdf)

    if not month_parts:
        return pd.DataFrame(columns=FEATURE_NAMES + [
            'true_state',
            'primary_event_distance',
            'secondary_transition_distance',
            'primary_event_exact',
            'secondary_transition_exact',
            'primary_event_type',
            'building','year','month','month_id'
        ])

    out = pd.concat(month_parts).sort_index()
    event_meta = TRAINING_EVENT_META[building]
    for c in [
        'primary_event_distance',
        'secondary_transition_distance',
        'primary_event_exact',
        'secondary_transition_exact',
        'primary_event_type',
    ]:
        out[c] = event_meta[c].reindex(out.index)

    out['building'] = building

    # Feature validity requires every engineered input to be finite. Target may remain NaN for pure inference rows.
    X = out[FEATURE_NAMES].replace([np.inf,-np.inf], np.nan)
    out['feature_valid'] = X.notna().all(axis=1)
    out = out.loc[out['feature_valid']].copy()

    out.to_pickle(cache_path, compression='gzip')
    return out


FEATURE_TABLES = {}
for b in BUILDING_NAMES:
    print('Context features:', b)
    FEATURE_TABLES[b] = build_building_feature_table(b)
    print('  rows:', len(FEATURE_TABLES[b]), 'months:', FEATURE_TABLES[b]['month_id'].nunique() if len(FEATURE_TABLES[b]) else 0)

feature_manifest = pd.DataFrame([
    {
        'building': b,
        'rows': len(df),
        'months': int(df['month_id'].nunique()) if len(df) else 0,
        'target_valid_rows': int(df['true_state'].notna().sum()) if len(df) else 0,
    }
    for b,df in FEATURE_TABLES.items()
])
feature_manifest.to_csv(TABLE_DIR / 'feature_manifest.csv', index=False)
display(feature_manifest)
print('Feature count:', len(FEATURE_NAMES))
print(FEATURE_NAMES)

In [ ]:
# CELL 9 — Final event-aware weighting scheme + paper-ready weight figure

FIXED_EVENT_WEIGHTS = (
    CFG.fixed_weight_ordinary,
    CFG.fixed_weight_secondary_pm30,
    CFG.fixed_weight_secondary_pm15,
    CFG.fixed_weight_secondary_exact,
    CFG.fixed_weight_primary_pm30,
    CFG.fixed_weight_primary_pm15,
    CFG.fixed_weight_primary_exact,
)

weight_scheme_table = pd.DataFrame([
    {'Point type':'Ordinary interval', 'Final fixed weight':CFG.fixed_weight_ordinary},
    {'Point type':'Secondary transition ±30 min', 'Final fixed weight':CFG.fixed_weight_secondary_pm30},
    {'Point type':'Secondary transition ±15 min', 'Final fixed weight':CFG.fixed_weight_secondary_pm15},
    {'Point type':'Secondary transition exact', 'Final fixed weight':CFG.fixed_weight_secondary_exact},
    {'Point type':'Primary event ±30 min', 'Final fixed weight':CFG.fixed_weight_primary_pm30},
    {'Point type':'Primary event ±15 min', 'Final fixed weight':CFG.fixed_weight_primary_pm15},
    {'Point type':'Primary event exact', 'Final fixed weight':CFG.fixed_weight_primary_exact},
])

weight_scheme_table.to_csv(
    TABLE_DIR / 'Table_Training_Weight_Scheme.csv',
    index=False,
)
display(weight_scheme_table)

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.bar(
    weight_scheme_table['Point type'],
    weight_scheme_table['Final fixed weight'],
)
ax.set_ylabel('Loss weight')
ax.set_title('Event-aware training weight hierarchy')
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(
    PLOT_PAPER_DIR / 'Figure_Training_Weight_Hierarchy.png',
    dpi=CFG.plot_dpi,
    bbox_inches='tight',
)
plt.show()

## Event-aware leave-one-system-out training

Each of seven systems is held out for testing. Of the six remaining systems, one provides validation data and five provide training data. Five independently initialized models are trained per outer fold.

Training minimizes the original weighted mean-squared-error objective,

\[
\mathcal L = \frac{\sum_i w_i(\hat y_i-y_i)^2}{\sum_i w_i},
\]

using the original five-stage curriculum. Stages 1–4 provide warm-up; final checkpoint selection takes place only in Stage 5. Event weights use the same principal-start/shutdown definitions subsequently applied in evaluation.

In [ ]:
# CELL 10 — Compact MLP, fold-local scaling, balanced sampling,
# and FINAL primary/secondary event-aware loss

class ScoreStatsMLP(nn.Module):
    def __init__(self, n_features=len(FEATURE_NAMES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, CFG.hidden1),
            nn.LayerNorm(CFG.hidden1),
            nn.Tanh(),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden1, CFG.hidden2),
            nn.Tanh(),
            nn.Linear(CFG.hidden2, CFG.hidden3),
            nn.Tanh(),
            nn.Linear(CFG.hidden3, 1),
        )
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            gain = nn.init.calculate_gain('tanh') if m.out_features != 1 else 1.0
            nn.init.xavier_uniform_(m.weight, gain=gain)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return torch.sigmoid(self.net(x).squeeze(-1))


def supervised_rows(buildings):
    parts = []
    for b in buildings:
        df = FEATURE_TABLES[b]
        if len(df):
            valid = (
                df['true_state'].notna()
                & df['primary_event_distance'].notna()
                & df['secondary_transition_distance'].notna()
            )
            parts.append(df.loc[valid].copy())
    return pd.concat(parts).sort_index() if parts else pd.DataFrame()


def fit_fold_scaler(train_df):
    scaler = RobustScaler(
        quantile_range=(25, 75),
        with_centering=True,
        with_scaling=True,
        unit_variance=False,
    )
    scaler.fit(train_df[FEATURE_NAMES].to_numpy(float))
    scale = scaler.scale_.copy()
    scale[~np.isfinite(scale) | (np.abs(scale) < 1e-9)] = 1.0
    return scaler.center_.astype(np.float32), scale.astype(np.float32)


def balanced_sample_weights(frame):
    """Equal expected mass per building, then per month within building."""
    tmp = frame[['building', 'month_id']].reset_index(drop=True)
    months_per_building = tmp.groupby('building')['month_id'].nunique().to_dict()
    group_sizes = tmp.groupby(['building', 'month_id']).size().to_dict()

    w = np.empty(len(tmp), dtype=np.float64)
    for i, (b, m) in enumerate(zip(tmp['building'], tmp['month_id'])):
        w[i] = 1.0 / (months_per_building[b] * group_sizes[(b, m)])

    w /= np.sum(w)
    return w


def weights_from_event_distances(primary_dist, secondary_dist, weights):
    (
        ordinary,
        sec30, sec15, sec_exact,
        pri30, pri15, pri_exact,
    ) = [float(v) for v in weights]

    w = torch.full_like(primary_dist, ordinary, dtype=torch.float32)

    # Moderate weights around secondary transitions.
    for dist_value, value in [(2, sec30), (1, sec15), (0, sec_exact)]:
        w = torch.maximum(
            w,
            torch.where(
                secondary_dist == dist_value,
                torch.as_tensor(value, device=w.device),
                torch.as_tensor(ordinary, device=w.device),
            ),
        )

    # Strongest weights around primary daily operational events.
    for dist_value, value in [(2, pri30), (1, pri15), (0, pri_exact)]:
        w = torch.maximum(
            w,
            torch.where(
                primary_dist == dist_value,
                torch.as_tensor(value, device=w.device),
                torch.as_tensor(ordinary, device=w.device),
            ),
        )

    return w


def weighted_event_mse(score, y, primary_dist, secondary_dist, weights):
    w = weights_from_event_distances(
        primary_dist,
        secondary_dist,
        weights,
    )
    return (
        torch.sum(w * (score - y) ** 2)
        / torch.clamp(torch.sum(w), min=1e-12)
    )


def selection_score(train_fixed, val_fixed):
    return float(
        val_fixed
        + CFG.selection_gap_lambda * max(0.0, val_fixed - train_fixed)
    )


def smoothed(values):
    x = [float(v) for v in values if np.isfinite(v)]
    if not x:
        return np.nan
    return float(np.mean(x[-CFG.smoothing_window:]))


def stage_tuple(stage_idx):
    row = CFG.curriculum[stage_idx]
    return float(row[0]), tuple(float(v) for v in row[1:])


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fold_split(test_building):
    i = BUILDING_NAMES.index(test_building)
    val_building = BUILDING_NAMES[(i + 1) % len(BUILDING_NAMES)]
    train_buildings = [
        b for b in BUILDING_NAMES
        if b not in {test_building, val_building}
    ]
    return train_buildings, val_building


def _normalized_arrays(frame, center, scale):
    X = frame[FEATURE_NAMES].to_numpy(np.float32)
    X = (
        X - np.asarray(center, np.float32)
    ) / np.asarray(scale, np.float32)

    y = pd.to_numeric(
        frame['true_state'],
        errors='coerce',
    ).to_numpy(np.float32)

    primary_dist = pd.to_numeric(
        frame['primary_event_distance'],
        errors='coerce',
    ).fillna(99).to_numpy(np.int64)

    secondary_dist = pd.to_numeric(
        frame['secondary_transition_distance'],
        errors='coerce',
    ).fillna(99).to_numpy(np.int64)

    return (
        X.astype(np.float32),
        y.astype(np.float32),
        primary_dist,
        secondary_dist,
    )


def _building_codes(frame):
    names = list(pd.unique(frame['building'].astype(str)))
    mapping = {b: i for i, b in enumerate(names)}
    codes = frame['building'].astype(str).map(mapping).to_numpy(np.int64)
    return codes, names


def prepare_fold_tensors(test_building):
    train_buildings, val_building = fold_split(test_building)

    train_df = supervised_rows(train_buildings)
    val_df = supervised_rows([val_building])

    if train_df.empty or val_df.empty:
        raise RuntimeError(f'{test_building}: empty train/validation data.')

    center, scale = fit_fold_scaler(train_df)

    Xtr, ytr, ptr, str_ = _normalized_arrays(train_df, center, scale)
    Xva, yva, pva, sva = _normalized_arrays(val_df, center, scale)

    tr_codes, tr_names = _building_codes(train_df)
    va_codes, va_names = _building_codes(val_df)

    sample_prob = balanced_sample_weights(train_df).astype(np.float32)

    fold = {
        'test_building': test_building,
        'val_building': val_building,
        'train_buildings': train_buildings,
        'center': center,
        'scale': scale,
        'train_df': train_df,
        'val_df': val_df,

        'X_train': torch.as_tensor(Xtr, dtype=torch.float32, device=DEVICE),
        'y_train': torch.as_tensor(ytr, dtype=torch.float32, device=DEVICE),
        'primary_dist_train': torch.as_tensor(ptr, dtype=torch.long, device=DEVICE),
        'secondary_dist_train': torch.as_tensor(str_, dtype=torch.long, device=DEVICE),
        'train_building_codes': torch.as_tensor(tr_codes, dtype=torch.long, device=DEVICE),
        'train_building_names': tr_names,
        'sample_prob': torch.as_tensor(sample_prob, dtype=torch.float32, device=DEVICE),

        'X_val': torch.as_tensor(Xva, dtype=torch.float32, device=DEVICE),
        'y_val': torch.as_tensor(yva, dtype=torch.float32, device=DEVICE),
        'primary_dist_val': torch.as_tensor(pva, dtype=torch.long, device=DEVICE),
        'secondary_dist_val': torch.as_tensor(sva, dtype=torch.long, device=DEVICE),
        'val_building_codes': torch.as_tensor(va_codes, dtype=torch.long, device=DEVICE),
        'val_building_names': va_names,
    }

    fold['n_train'] = int(len(train_df))
    fold['n_val'] = int(len(val_df))
    fold['train_batches_per_epoch'] = int(
        math.ceil(max(1, fold['n_train']) / CFG.batch_size)
    )

    print(
        f"Prepared fold {test_building}: train={fold['n_train']:,}, "
        f"val={fold['n_val']:,}, features={len(FEATURE_NAMES)}, device={DEVICE}"
    )
    return fold


@torch.inference_mode()
def fixed_macro_loss_tensors(
    model, X, y, primary_dist, secondary_dist,
    building_codes, n_buildings
):
    model.eval()
    num = torch.zeros(n_buildings, dtype=torch.float64, device=X.device)
    den = torch.zeros(n_buildings, dtype=torch.float64, device=X.device)

    bs = int(CFG.eval_batch_size)

    for start in range(0, len(X), bs):
        stop = min(start + bs, len(X))

        xb = X[start:stop]
        yb = y[start:stop]
        pb = primary_dist[start:stop]
        sb = secondary_dist[start:stop]
        cb = building_codes[start:stop]

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16,
            enabled=(AMP_ENABLED and CFG.use_amp),
        ):
            score = model(xb)

        w = weights_from_event_distances(pb, sb, FIXED_EVENT_WEIGHTS)
        se = (w * (score.float() - yb.float()) ** 2).to(torch.float64)
        wd = w.to(torch.float64)

        num.scatter_add_(0, cb, se)
        den.scatter_add_(0, cb, wd)

    valid = den > 0
    if not torch.any(valid):
        return np.nan

    return float(torch.mean(num[valid] / den[valid]).item())


def _cpu_state_dict(model):
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }


def _make_sampling_generator(seed):
    g = torch.Generator(device='cuda') if DEVICE.type == 'cuda' else torch.Generator()
    g.manual_seed(int(seed))
    return g


print(
    f'Event-aware training: device={DEVICE}, '
    f'batch={CFG.batch_size}, eval_batch={CFG.eval_batch_size}'
)

In [ ]:
# CELL 11 — GPU-parallel multi-seed LOBO training
# GUARANTEE:
#   * every seed reaches Stage 5;
#   * Stages 1-4 are warm-up/curriculum only;
#   * only Stage-5 checkpoints are eligible for final model selection.

TRAINING_LOGIC_VERSION = 'event_aware_daily_schedule_stage5_eventrulefix_v3'


def _checkpoint_is_compatible(meta_path):
    """
    Reuse a completed seed only if it was produced by this exact
    Stage-5-guaranteed training logic and its final checkpoint is Stage 5.
    """
    try:
        with open(meta_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)

        return bool(
            meta.get('training_logic_version') == TRAINING_LOGIC_VERSION
            and int(meta.get('best_stage', -1)) == len(CFG.curriculum)
            and bool(meta.get('all_stages_reached', False))
        )
    except Exception:
        return False


def _new_seed_state(test_building, fold, seed, stream=None):
    fold_dir = MODEL_DIR / test_building
    fold_dir.mkdir(parents=True, exist_ok=True)

    model_path = fold_dir / f'seed_{seed}_best.pt'
    history_path = fold_dir / f'seed_{seed}_history.csv'
    meta_path = fold_dir / f'seed_{seed}_meta.json'

    if (
        CFG.resume_existing
        and model_path.exists()
        and history_path.exists()
        and meta_path.exists()
        and _checkpoint_is_compatible(meta_path)
    ):
        return {
            'seed': int(seed),
            'reuse': True,
            'model_path': model_path,
            'history_path': history_path,
            'meta_path': meta_path,
        }

    if (
        CFG.resume_existing
        and model_path.exists()
        and meta_path.exists()
        and not _checkpoint_is_compatible(meta_path)
    ):
        print(
            f'{test_building} seed={seed}: existing checkpoint is not '
            'Stage-5-guaranteed; retraining this seed.'
        )

    set_all_seeds(seed)
    model = ScoreStatsMLP().to(DEVICE)

    stage_idx = 0
    lr, opt_weights = stage_tuple(stage_idx)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=CFG.weight_decay,
    )

    scaler_amp = torch.cuda.amp.GradScaler(
        enabled=(AMP_ENABLED and CFG.use_amp)
    )

    return {
        'seed': int(seed),
        'reuse': False,
        'stream': stream,
        'generator': _make_sampling_generator(seed),

        'model': model,
        'optimizer': optimizer,
        'scaler_amp': scaler_amp,

        'model_path': model_path,
        'history_path': history_path,
        'meta_path': meta_path,

        'stage_idx': stage_idx,
        'opt_weights': opt_weights,
        'history': [],

        # Best checkpoint inside the CURRENT stage.
        'stage_best': None,
        'stage_best_score': np.inf,

        # FINAL checkpoint pool: Stage 5 ONLY.
        'final_best': None,
        'final_best_score': np.inf,
        'final_best_epoch': -1,
        'final_no_improve_count': 0,

        'stage_start_epoch': 1,
        'stage_best_train_sm': np.inf,
        'plateau_count': 0,
        'val_rise_count': 0,
        'prev_stage_val_sm': None,

        'reached_stage5': False,

        'epoch': 0,
        'stop_reason': 'not_stopped',
        'stopped': False,

        'pending_opt_sum': None,
        'pending_n_batch': 0,
    }


def _launch_one_seed_epoch(state, fold):
    """
    Enqueue one complete optimization epoch for one seed.
    No .item() calls occur here, so CUDA streams can overlap seed work.
    """
    model = state['model']
    optimizer = state['optimizer']
    scaler_amp = state['scaler_amp']
    generator = state['generator']

    model.train()

    n_train = fold['n_train']
    if n_train <= 0:
        raise RuntimeError('Empty training tensor.')

    sampled_idx = torch.multinomial(
        fold['sample_prob'],
        num_samples=n_train,
        replacement=True,
        generator=generator,
    )

    loss_sum = torch.zeros(
        (),
        dtype=torch.float32,
        device=DEVICE,
    )
    n_batch = 0

    for start in range(0, n_train, CFG.batch_size):
        stop = min(start + CFG.batch_size, n_train)
        idx = sampled_idx[start:stop]

        xb = fold['X_train'][idx]
        yb = fold['y_train'][idx]
        pb = fold['primary_dist_train'][idx]
        sb = fold['secondary_dist_train'][idx]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16,
            enabled=(AMP_ENABLED and CFG.use_amp),
        ):
            score = model(xb)

            loss = weighted_event_mse(
                score.float(),
                yb.float(),
                pb,
                sb,
                state['opt_weights'],
            )

        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CFG.grad_clip_norm,
        )

        scaler_amp.step(optimizer)
        scaler_amp.update()

        loss_sum = loss_sum + loss.detach().float()
        n_batch += 1

    state['pending_opt_sum'] = loss_sum
    state['pending_n_batch'] = n_batch


def _restore_stage_best(state):
    """
    Restore the best checkpoint of the current warm-up stage before
    progressing to the next curriculum stage.
    """
    if state['stage_best'] is None:
        return None

    state['model'].load_state_dict(
        state['stage_best']['model']
    )
    state['optimizer'].load_state_dict(
        state['stage_best']['optimizer']
    )

    return int(
        state['stage_best']['epoch']
    )


def _advance_stage(state, epoch, reason, test_building):
    """
    Advance one curriculum stage. This function is called only for Stages 1-4.
    """
    restored_epoch = _restore_stage_best(state)

    state['stage_idx'] += 1

    if state['stage_idx'] >= len(CFG.curriculum):
        raise RuntimeError('Attempted to advance beyond final curriculum stage.')

    lr, new_weights = stage_tuple(state['stage_idx'])
    state['opt_weights'] = new_weights

    for pg in state['optimizer'].param_groups:
        pg['lr'] = lr

    state['stage_start_epoch'] = epoch + 1
    state['stage_best'] = None
    state['stage_best_score'] = np.inf
    state['stage_best_train_sm'] = np.inf
    state['plateau_count'] = 0
    state['val_rise_count'] = 0
    state['prev_stage_val_sm'] = None

    # Reset the final-stage early-stopping counter on entry to Stage 5.
    if state['stage_idx'] == len(CFG.curriculum) - 1:
        state['reached_stage5'] = True
        state['final_no_improve_count'] = 0

    msg = (
        f'restored stage-best epoch {restored_epoch}'
        if restored_epoch is not None
        else 'continued from current weights'
    )

    print(
        f'--> {test_building} seed={state["seed"]}: '
        f'advance to stage {state["stage_idx"] + 1} '
        f'[{reason}; {msg}] '
        f'LR={lr:.1e}, weights={new_weights}'
    )


def _monitor_and_update_seed(state, fold, test_building):
    """
    Fixed-loss monitoring + guaranteed curriculum progression.

    Stages 1-4:
        advance on plateau OR validation rise OR stage epoch cap.

    Stage 5:
        ONLY Stage-5 checkpoints are eligible for final selection.
        Stop after final_stage_patience epochs without a better selection score.
    """
    model = state['model']
    optimizer = state['optimizer']

    state['epoch'] += 1
    epoch = state['epoch']
    stage_idx = state['stage_idx']
    final_stage_idx = len(CFG.curriculum) - 1

    train_opt = (
        float(state['pending_opt_sum'].item())
        / max(state['pending_n_batch'], 1)
    )

    train_fixed = fixed_macro_loss_tensors(
        model,
        fold['X_train'],
        fold['y_train'],
        fold['primary_dist_train'],
        fold['secondary_dist_train'],
        fold['train_building_codes'],
        len(fold['train_building_names']),
    )

    val_fixed = fixed_macro_loss_tensors(
        model,
        fold['X_val'],
        fold['y_val'],
        fold['primary_dist_val'],
        fold['secondary_dist_val'],
        fold['val_building_codes'],
        len(fold['val_building_names']),
    )

    sel = selection_score(
        train_fixed,
        val_fixed,
    )

    lr_now = float(
        optimizer.param_groups[0]['lr']
    )

    state['history'].append({
        'epoch': epoch,
        'stage': stage_idx + 1,
        'lr': lr_now,

        'w_ordinary': state['opt_weights'][0],
        'w_secondary_pm30': state['opt_weights'][1],
        'w_secondary_pm15': state['opt_weights'][2],
        'w_secondary_exact': state['opt_weights'][3],
        'w_primary_pm30': state['opt_weights'][4],
        'w_primary_pm15': state['opt_weights'][5],
        'w_primary_exact': state['opt_weights'][6],

        'train_optimization_mse': train_opt,
        'train_fixed_mse': train_fixed,
        'val_fixed_mse': val_fixed,
        'selection_score': sel,
    })

    stage_rows = [
        r for r in state['history']
        if r['stage'] == stage_idx + 1
    ]

    tr_sm = smoothed(
        [r['train_fixed_mse'] for r in stage_rows]
    )

    va_sm = smoothed(
        [r['val_fixed_mse'] for r in stage_rows]
    )

    state['history'][-1]['stage_train_fixed_smooth'] = tr_sm
    state['history'][-1]['stage_val_fixed_smooth'] = va_sm

    # --------------------------------------------------------------
    # Best checkpoint in CURRENT stage (used for warm-up restoration).
    # --------------------------------------------------------------
    if sel < state['stage_best_score']:
        state['stage_best_score'] = float(sel)

        state['stage_best'] = {
            'model': _cpu_state_dict(model),
            'optimizer': copy.deepcopy(
                optimizer.state_dict()
            ),
            'stage_idx': int(stage_idx),
            'epoch': int(epoch),
            'train_fixed': float(train_fixed),
            'val_fixed': float(val_fixed),
            'selection_score': float(sel),
        }

    # --------------------------------------------------------------
    # FINAL candidate pool: Stage 5 ONLY.
    # --------------------------------------------------------------
    final_improved = False

    if stage_idx == final_stage_idx:
        state['reached_stage5'] = True

        if sel < state['final_best_score']:
            state['final_best_score'] = float(sel)
            state['final_best_epoch'] = int(epoch)

            state['final_best'] = {
                'model': _cpu_state_dict(model),
                'stage_idx': int(stage_idx),
                'epoch': int(epoch),
                'train_fixed': float(train_fixed),
                'val_fixed': float(val_fixed),
                'selection_score': float(sel),
            }

            state['final_no_improve_count'] = 0
            final_improved = True
        else:
            state['final_no_improve_count'] += 1

    # --------------------------------------------------------------
    # Warm-up progression diagnostics.
    # --------------------------------------------------------------
    if np.isfinite(state['stage_best_train_sm']):
        rel = (
            state['stage_best_train_sm'] - tr_sm
        ) / max(
            abs(state['stage_best_train_sm']),
            1e-12,
        )
    else:
        rel = np.inf

    if (
        not np.isfinite(state['stage_best_train_sm'])
        or rel > CFG.train_plateau_min_rel_improvement
    ):
        state['stage_best_train_sm'] = float(tr_sm)
        state['plateau_count'] = 0
    else:
        state['plateau_count'] += 1

    if (
        state['prev_stage_val_sm'] is not None
        and va_sm > state['prev_stage_val_sm']
    ):
        state['val_rise_count'] += 1
    else:
        state['val_rise_count'] = 0

    state['prev_stage_val_sm'] = float(va_sm)

    epochs_in_stage = (
        epoch
        - state['stage_start_epoch']
        + 1
    )

    plateau_trigger = bool(
        stage_idx < final_stage_idx
        and epochs_in_stage >= CFG.min_epochs_per_stage
        and state['plateau_count'] >= CFG.train_plateau_patience
    )

    val_rise_trigger = bool(
        stage_idx < final_stage_idx
        and epochs_in_stage >= CFG.min_epochs_per_stage
        and state['val_rise_count'] >= CFG.val_rise_patience
    )

    stage_cap_trigger = bool(
        stage_idx < final_stage_idx
        and epochs_in_stage >= CFG.max_epochs_per_stage
    )

    final_stop_trigger = bool(
        stage_idx == final_stage_idx
        and epochs_in_stage >= CFG.min_epochs_per_stage
        and state['final_no_improve_count'] >= CFG.final_stage_patience
    )

    if val_rise_trigger:
        reason = 'validation_rise'
    elif plateau_trigger:
        reason = 'train_plateau'
    elif stage_cap_trigger:
        reason = 'stage_epoch_cap'
    elif final_stop_trigger:
        reason = 'final_stage_no_improvement'
    else:
        reason = ''

    state['history'][-1]['epochs_in_stage'] = int(
        epochs_in_stage
    )
    state['history'][-1]['plateau_count'] = int(
        state['plateau_count']
    )
    state['history'][-1]['val_rise_count'] = int(
        state['val_rise_count']
    )
    state['history'][-1]['final_no_improve_count'] = int(
        state['final_no_improve_count']
    )
    state['history'][-1]['final_candidate'] = bool(
        stage_idx == final_stage_idx
    )
    state['history'][-1]['final_improved'] = bool(
        final_improved
    )
    state['history'][-1]['trigger_reason'] = reason

    should_print = bool(
        epoch == 1
        or epoch % 5 == 0
        or plateau_trigger
        or val_rise_trigger
        or stage_cap_trigger
        or final_stop_trigger
    )

    if should_print:
        print(
            f'{test_building} seed={state["seed"]} '
            f'e={epoch:03d} stage={stage_idx + 1} '
            f'stage_e={epochs_in_stage:02d} '
            f'lr={lr_now:.1e} '
            f'opt={train_opt:.5f} '
            f'train_fixed={train_fixed:.5f} '
            f'val_fixed={val_fixed:.5f} '
            f'sel={sel:.5f} '
            f'train_sm={tr_sm:.5f} '
            f'val_sm={va_sm:.5f} '
            f'final_wait={state["final_no_improve_count"]}'
        )

    # --------------------------------------------------------------
    # Stages 1-4: MUST advance, never finalize.
    # --------------------------------------------------------------
    if stage_idx < final_stage_idx:
        if (
            plateau_trigger
            or val_rise_trigger
            or stage_cap_trigger
        ):
            _advance_stage(
                state,
                epoch,
                reason,
                test_building,
            )

    # --------------------------------------------------------------
    # Stage 5: stop and later restore best Stage-5 checkpoint.
    # --------------------------------------------------------------
    else:
        if final_stop_trigger:
            state['stop_reason'] = (
                'final_stage_early_stop'
            )
            state['stopped'] = True

            print(
                f'--> {test_building} seed={state["seed"]}: '
                f'Stage 5 complete; stop after '
                f'{CFG.final_stage_patience} epochs without improvement.'
            )

    # --------------------------------------------------------------
    # Absolute safety ceiling.
    # It may finalize only after Stage 5 has actually been reached.
    # --------------------------------------------------------------
    if (
        epoch >= CFG.max_epochs
        and not state['stopped']
    ):
        if (
            stage_idx == final_stage_idx
            and state['final_best'] is not None
        ):
            state['stop_reason'] = (
                'max_epochs_after_stage5'
            )
            state['stopped'] = True

            print(
                f'--> {test_building} seed={state["seed"]}: '
                f'hit max_epochs={CFG.max_epochs} after reaching Stage 5.'
            )
        else:
            raise RuntimeError(
                f'{test_building} seed={state["seed"]}: '
                f'hit max_epochs={CFG.max_epochs} before a valid '
                'Stage-5 checkpoint existed. This violates the '
                'guaranteed curriculum.'
            )


def _finalize_seed_state(state, fold, test_building):
    if state.get('reuse', False):
        print(
            'Reuse compatible Stage-5 checkpoint:',
            state['model_path'],
        )

        return {
            'test_building': test_building,
            'val_building': fold['val_building'],
            'seed': int(state['seed']),
            'model_path': str(state['model_path']),
            'history_path': str(state['history_path']),
        }

    final_stage_idx = (
        len(CFG.curriculum) - 1
    )

    if not state['reached_stage5']:
        raise RuntimeError(
            f'{test_building} seed={state["seed"]}: '
            'finalization attempted before Stage 5.'
        )

    if state['final_best'] is None:
        raise RuntimeError(
            f'{test_building} seed={state["seed"]}: '
            'no Stage-5 checkpoint was recorded.'
        )

    if (
        int(state['final_best']['stage_idx'])
        != final_stage_idx
    ):
        raise RuntimeError(
            f'{test_building} seed={state["seed"]}: '
            'final checkpoint is not from Stage 5.'
        )

    model = state['model']

    model.load_state_dict(
        state['final_best']['model']
    )

    history_df = pd.DataFrame(
        state['history']
    )

    history_df.to_csv(
        state['history_path'],
        index=False,
    )

    payload = {
        'state_dict': _cpu_state_dict(model),

        'test_building': test_building,
        'val_building': fold['val_building'],
        'train_buildings': fold['train_buildings'],
        'seed': int(state['seed']),

        'best_epoch': int(
            state['final_best_epoch']
        ),
        'best_stage': int(
            state['final_best']['stage_idx']
        ) + 1,

        'best_train_fixed_mse': float(
            state['final_best']['train_fixed']
        ),
        'best_val_fixed_mse': float(
            state['final_best']['val_fixed']
        ),
        'best_selection_score': float(
            state['final_best_score']
        ),

        'stop_reason': state['stop_reason'],
        'all_stages_reached': True,

        'stage5_start_epoch': int(
            history_df.loc[
                history_df['stage'] == len(CFG.curriculum),
                'epoch',
            ].min()
        ),

        'total_epochs_run': int(
            state['epoch']
        ),

        'feature_names': FEATURE_NAMES,
        'scaler_center': fold['center'].tolist(),
        'scaler_scale': fold['scale'].tolist(),

        'config': asdict(CFG),
        'training_logic_version': TRAINING_LOGIC_VERSION,
        'building_set': list(BUILDING_NAMES),
    }

    if CFG.save_models:
        torch.save(
            payload,
            state['model_path'],
        )

    with open(
        state['meta_path'],
        'w',
        encoding='utf-8',
    ) as f:
        json.dump(
            {
                k: v
                for k, v in payload.items()
                if k not in {
                    'state_dict',
                    'scaler_center',
                    'scaler_scale',
                }
            },
            f,
            indent=2,
            default=str,
        )

    print(
        f'FINAL {test_building} seed={state["seed"]}: '
        f'Stage-5 best epoch={state["final_best_epoch"]}, '
        f'train_fixed={state["final_best"]["train_fixed"]:.5f}, '
        f'val_fixed={state["final_best"]["val_fixed"]:.5f}, '
        f'sel={state["final_best_score"]:.5f}, '
        f'stop={state["stop_reason"]}'
    )

    result = {
        'test_building': test_building,
        'val_building': fold['val_building'],
        'seed': int(state['seed']),
        'model_path': str(state['model_path']),
        'history_path': str(state['history_path']),
    }

    del state['model']
    del state['optimizer']
    del state['scaler_amp']

    return result


def train_fold_parallel_seeds(test_building):
    fold = prepare_fold_tensors(
        test_building
    )

    use_parallel_streams = bool(
        DEVICE.type == 'cuda'
        and CFG.parallel_seed_training
        and len(CFG.lobo_seeds) > 1
    )

    if use_parallel_streams:
        streams = {
            int(seed): torch.cuda.Stream(
                device=DEVICE
            )
            for seed in CFG.lobo_seeds
        }

        print(
            f'{test_building}: training {len(CFG.lobo_seeds)} seeds '
            'with independent CUDA streams.'
        )
    else:
        streams = {
            int(seed): None
            for seed in CFG.lobo_seeds
        }

        print(
            f'{test_building}: CUDA parallel streams unavailable/disabled; '
            'seed epochs will run sequentially.'
        )

    states = []
    results = []

    for seed in CFG.lobo_seeds:
        st = _new_seed_state(
            test_building,
            fold,
            int(seed),
            stream=streams[int(seed)],
        )

        if st.get('reuse', False):
            results.append(
                _finalize_seed_state(
                    st,
                    fold,
                    test_building,
                )
            )
        else:
            states.append(st)

    # Round-robin across active seeds.
    while states:
        active = [
            s
            for s in states
            if not s['stopped']
        ]

        if not active:
            break

        # Launch one epoch for each active seed.
        for st in active:
            if use_parallel_streams:
                with torch.cuda.stream(
                    st['stream']
                ):
                    _launch_one_seed_epoch(
                        st,
                        fold,
                    )
            else:
                _launch_one_seed_epoch(
                    st,
                    fold,
                )

        if use_parallel_streams:
            torch.cuda.synchronize()

        for st in list(active):
            _monitor_and_update_seed(
                st,
                fold,
                test_building,
            )

            if st['stopped']:
                results.append(
                    _finalize_seed_state(
                        st,
                        fold,
                        test_building,
                    )
                )

                states.remove(st)

        gc.collect()

    del fold

    gc.collect()

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    return results

In [ ]:
# CELL 12 — Run genuine 7-building LOBO
# Outer held-out building = TEST.
# Every trained seed must finish with a Stage-5-selected checkpoint.

TRAINING_RUNS = []

folds = list(
    BUILDING_NAMES
)

if CFG.max_folds is not None:
    folds = folds[
        :int(CFG.max_folds)
    ]

if CFG.run_training:
    for test_building in folds:
        train_buildings, val_building = (
            fold_split(
                test_building
            )
        )

        print(
            '\n'
            + '=' * 100
        )

        print(
            'OUTER TEST:', test_building,
            '| VALIDATION:', val_building,
            '| TRAIN:', train_buildings,
        )

        TRAINING_RUNS.extend(
            train_fold_parallel_seeds(
                test_building
            )
        )
else:
    print(
        'Training disabled; compatible Stage-5 checkpoints '
        'in the final directory will be used.'
    )


# --------------------------------------------------------------
# Training-summary table: all final models MUST be Stage 5.
# --------------------------------------------------------------
training_summary_rows = []

for test_building in folds:
    for seed in CFG.lobo_seeds:
        meta_path = (
            MODEL_DIR
            / test_building
            / f'seed_{seed}_meta.json'
        )

        if not meta_path.exists():
            raise FileNotFoundError(
                f'Missing seed metadata: {meta_path}'
            )

        with open(
            meta_path,
            'r',
            encoding='utf-8',
        ) as f:
            meta = json.load(f)

        if (
            meta.get('training_logic_version')
            != TRAINING_LOGIC_VERSION
        ):
            raise RuntimeError(
                f'Incompatible training logic for '
                f'{test_building} seed={seed}.'
            )

        if int(
            meta.get(
                'best_stage',
                -1,
            )
        ) != len(CFG.curriculum):
            raise RuntimeError(
                f'{test_building} seed={seed}: '
                'final checkpoint is not Stage 5.'
            )

        if not bool(
            meta.get(
                'all_stages_reached',
                False,
            )
        ):
            raise RuntimeError(
                f'{test_building} seed={seed}: '
                'not all curriculum stages were reached.'
            )

        training_summary_rows.append({
            'test_building':
                test_building,

            'validation_building':
                meta[
                    'val_building'
                ],

            'seed':
                int(seed),

            'stage5_start_epoch':
                int(
                    meta[
                        'stage5_start_epoch'
                    ]
                ),

            'best_epoch':
                int(
                    meta[
                        'best_epoch'
                    ]
                ),

            'best_stage':
                int(
                    meta[
                        'best_stage'
                    ]
                ),

            'total_epochs_run':
                int(
                    meta[
                        'total_epochs_run'
                    ]
                ),

            'best_train_fixed_mse':
                float(
                    meta[
                        'best_train_fixed_mse'
                    ]
                ),

            'best_val_fixed_mse':
                float(
                    meta[
                        'best_val_fixed_mse'
                    ]
                ),

            'best_selection_score':
                float(
                    meta[
                        'best_selection_score'
                    ]
                ),

            'stop_reason':
                meta[
                    'stop_reason'
                ],

            'hit_global_max_epochs':
                bool(
                    meta[
                        'stop_reason'
                    ]
                    == 'max_epochs_after_stage5'
                ),
        })


training_summary_df = pd.DataFrame(
    training_summary_rows
)

training_summary_df.to_csv(
    TABLE_DIR
    / 'Table_Training_Stage5_Summary.csv',
    index=False,
)

display(
    training_summary_df
)

assert (
    training_summary_df[
        'best_stage'
    ]
    == len(CFG.curriculum)
).all()

print(
    'Stage-5 audit passed: '
    f'{len(training_summary_df)} / '
    f'{len(folds) * len(CFG.lobo_seeds)} '
    'final seed models were selected from Stage 5.'
)

## Held-out inference and schedule evaluation

For the excluded system, predictions from the five selected models are averaged and converted to a binary state using a fixed threshold of **0.5**, without target-label calibration. The decoder applies the original isolated-flip cleanup and principal-event definitions, including the already-active-at-04:00 case.

Interval, daily-event, operating-regime, and complete-month metrics are calculated under the original truth-coverage eligibility rules. Missing reference coverage is not scored as an operating-state error.

In [ ]:
# CELL 13 — Fixed-threshold held-out inference and evaluation utilities


def load_model_checkpoint(path):
    payload = torch.load(
        path,
        map_location=DEVICE,
        weights_only=False,
    )
    model = ScoreStatsMLP(
        len(payload['feature_names'])
    ).to(DEVICE)
    model.load_state_dict(payload['state_dict'])
    model.eval()
    return model, payload


@torch.inference_mode()
def predict_feature_frame(model, payload, frame):
    X = frame[payload['feature_names']].to_numpy(np.float32)
    center = np.asarray(payload['scaler_center'], dtype=np.float32)
    scale = np.asarray(payload['scaler_scale'], dtype=np.float32)
    X = (X - center) / scale

    scores = []
    for i in range(0, len(X), CFG.eval_batch_size):
        xb = torch.as_tensor(
            X[i:i + CFG.eval_batch_size],
            dtype=torch.float32,
            device=DEVICE,
        )
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16,
            enabled=(DEVICE.type == 'cuda' and CFG.use_amp),
        ):
            sb = model(xb)
        scores.append(sb.float().cpu().numpy())

    return pd.Series(
        np.concatenate(scores),
        index=frame.index,
        dtype=float,
    )


def threshold_state(score_series, threshold=None):
    """Final fixed-threshold decoder."""
    if threshold is None:
        threshold = CFG.decision_threshold
    s = pd.to_numeric(score_series, errors='coerce')
    out = pd.Series(np.nan, index=s.index, dtype=float)
    m = s.notna()
    out.loc[m] = (s.loc[m] >= float(threshold)).astype(float)
    return out


def clean_isolated_binary_flips(state):
    s = pd.to_numeric(state, errors='coerce').astype(float).sort_index()
    if len(s) < 3:
        return s.copy()

    prev = s.shift(1)
    nxt = s.shift(-1)
    idx = s.index.to_series()

    contiguous_prev = idx.diff() == pd.Timedelta(minutes=CFG.dt_minutes)
    contiguous_next = (idx.shift(-1) - idx) == pd.Timedelta(minutes=CFG.dt_minutes)

    flip = (
        s.notna() & prev.notna() & nxt.notna()
        & contiguous_prev & contiguous_next
        & prev.eq(nxt) & s.ne(prev)
    )

    out = s.copy()
    out.loc[flip] = prev.loc[flip]
    return out


def time_to_minutes(ts):
    if pd.isna(ts):
        return np.nan
    ts = pd.Timestamp(ts)
    return float(ts.hour * 60 + ts.minute + ts.second / 60.0)


def minutes_to_hhmm(v):
    if not np.isfinite(v):
        return ''
    v = float(v) % 1440.0
    h = int(v // 60)
    m = int(round(v - h * 60))
    if m == 60:
        h = (h + 1) % 24
        m = 0
    return f'{h:02d}:{m:02d}'


def _metric_mask(mask, n):
    if mask is None:
        return np.ones(int(n), dtype=bool)
    m = pd.Series(mask).fillna(False).to_numpy(dtype=bool)
    if len(m) != int(n):
        raise ValueError(f'Evaluation-mask length {len(m)} != data length {n}')
    return m


def event_detection_stats(true_times, pred_times, prefix, eval_mask=None):
    t = pd.Series(true_times).notna().to_numpy()
    p = pd.Series(pred_times).notna().to_numpy()
    m = _metric_mask(eval_mask, len(t))
    t = t[m]
    p = p[m]

    tp = int(np.sum(t & p))
    fp = int(np.sum(~t & p))
    fn = int(np.sum(t & ~p))

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0
        else np.nan
    )

    return {
        f'{prefix}_scored_days': int(len(t)),
        f'{prefix}_tp_days': tp,
        f'{prefix}_false_days': fp,
        f'{prefix}_missed_days': fn,
        f'{prefix}_precision': precision,
        f'{prefix}_recall': recall,
        f'{prefix}_f1': f1,
    }


def matched_timing_stats(true_times, pred_times, prefix, eval_mask=None):
    t = pd.to_datetime(pd.Series(true_times), errors='coerce')
    p = pd.to_datetime(pd.Series(pred_times), errors='coerce')
    m = _metric_mask(eval_mask, len(t))
    both = t.notna() & p.notna() & m

    if not both.any():
        return {
            f'{prefix}_matched_n': 0,
            f'{prefix}_mae_min': np.nan,
            f'{prefix}_median_ae_min': np.nan,
            f'{prefix}_rmse_min': np.nan,
            f'{prefix}_bias_min': np.nan,
            f'{prefix}_q90_ae_min': np.nan,
            f'{prefix}_within15': np.nan,
            f'{prefix}_within30': np.nan,
        }

    err = (
        p.loc[both].reset_index(drop=True)
        - t.loc[both].reset_index(drop=True)
    ).dt.total_seconds() / 60.0
    ae = err.abs()

    return {
        f'{prefix}_matched_n': int(len(err)),
        f'{prefix}_mae_min': float(ae.mean()),
        f'{prefix}_median_ae_min': float(ae.median()),
        f'{prefix}_rmse_min': float(np.sqrt(np.mean(np.square(err)))),
        f'{prefix}_bias_min': float(err.mean()),
        f'{prefix}_q90_ae_min': float(ae.quantile(0.90)),
        f'{prefix}_within15': float((ae <= CFG.timing_tolerance_15_min).mean()),
        f'{prefix}_within30': float((ae <= CFG.timing_tolerance_30_min).mean()),
    }


def schedule_distribution_stats(series, prefix):
    vals = np.array(
        [time_to_minutes(x) for x in series if pd.notna(x)],
        dtype=float,
    )
    vals = vals[np.isfinite(vals)]

    out = {f'{prefix}_n': int(len(vals))}
    names = ['mean','std','q10','q25','median','q75','q90','iqr']

    if not len(vals):
        for name in names:
            out[f'{prefix}_{name}_min'] = np.nan
        for name in ['q10','q25','median','q75','q90']:
            out[f'{prefix}_{name}_hhmm'] = ''
        return out

    q10, q25, med, q75, q90 = np.quantile(vals, [0.10,0.25,0.50,0.75,0.90])
    out.update({
        f'{prefix}_mean_min': float(np.mean(vals)),
        f'{prefix}_std_min': float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        f'{prefix}_q10_min': float(q10),
        f'{prefix}_q25_min': float(q25),
        f'{prefix}_median_min': float(med),
        f'{prefix}_q75_min': float(q75),
        f'{prefix}_q90_min': float(q90),
        f'{prefix}_iqr_min': float(q75 - q25),
        f'{prefix}_q10_hhmm': minutes_to_hhmm(q10),
        f'{prefix}_q25_hhmm': minutes_to_hhmm(q25),
        f'{prefix}_median_hhmm': minutes_to_hhmm(med),
        f'{prefix}_q75_hhmm': minutes_to_hhmm(q75),
        f'{prefix}_q90_hhmm': minutes_to_hhmm(q90),
    })
    return out


def month_expected_index(year, month):
    idx = load_raw.index[season_mask(load_raw.index)]
    mask = (idx.year == int(year)) & (idx.month == int(month))
    return idx[mask]


def month_qc(feature_month, truth_state, year, month):
    expected = month_expected_index(year, month)
    expected_n = int(len(expected))
    feature_valid_n = int(len(feature_month.index.unique()))
    feature_coverage = feature_valid_n / expected_n if expected_n else np.nan

    truth_reg = pd.to_numeric(truth_state, errors='coerce').reindex(expected)
    truth_valid_n = int(truth_reg.notna().sum())
    truth_coverage = truth_valid_n / expected_n if expected_n else np.nan

    m0 = pd.Timestamp(year=int(year), month=int(month), day=1)
    mend = m0 + pd.offsets.MonthEnd(1) + pd.Timedelta(hours=23, minutes=45)

    complete = bool(
        expected_n > 0
        and len(expected) > 0
        and expected.min() <= m0
        and expected.max() >= mend
    )
    coverage_eligible = bool(
        np.isfinite(feature_coverage)
        and np.isfinite(truth_coverage)
        and feature_coverage >= CFG.min_month_interval_coverage
        and truth_coverage >= CFG.min_month_truth_coverage
    )

    return {
        'expected_interval_n': expected_n,
        'feature_interval_n': feature_valid_n,
        'truth_interval_n': truth_valid_n,
        'feature_interval_coverage': feature_coverage,
        'truth_interval_coverage': truth_coverage,
        # Backward-compatible alias used by some figures/tables.
        'month_interval_coverage': feature_coverage,
        'complete_calendar_month': complete,
        'coverage_eligible': coverage_eligible,
        'evaluation_eligible': coverage_eligible,
        'primary_evaluation_eligible': bool(coverage_eligible and complete),
    }


def _analysis_dates_index():
    dates = []
    for season_start, season_end in SEASONS.values():
        dates.extend(
            pd.date_range(
                pd.Timestamp(season_start).normalize(),
                pd.Timestamp(season_end).normalize(),
                freq='D',
            )
        )
    return pd.DatetimeIndex(pd.unique(pd.DatetimeIndex(dates)))


ANALYSIS_DATES = _analysis_dates_index()


def build_daily_comparison(true_state, pred_state):
    true_daily = build_daily_operational_table(true_state)
    pred_daily = build_daily_operational_table(pred_state)

    true_daily = true_daily.loc[
        pd.to_datetime(true_daily['date']).dt.normalize().isin(ANALYSIS_DATES)
    ].copy()
    pred_daily = pred_daily.loc[
        pd.to_datetime(pred_daily['date']).dt.normalize().isin(ANALYSIS_DATES)
    ].copy()

    true_daily = true_daily.rename(columns={
        'turn_on_time':'true_turn_on',
        'turn_off_time':'true_turn_off',
        'day_status':'true_day_status',
        'always_on':'true_always_on',
        'always_off':'true_always_off',
        'already_on_at_04':'true_already_on_at_04',
        'day_state_coverage':'true_day_state_coverage',
        'n_raw_transitions':'true_raw_transitions',
    })
    pred_daily = pred_daily.rename(columns={
        'turn_on_time':'pred_turn_on',
        'turn_off_time':'pred_turn_off',
        'day_status':'pred_day_status',
        'always_on':'pred_always_on',
        'always_off':'pred_always_off',
        'already_on_at_04':'pred_already_on_at_04',
        'day_state_coverage':'pred_day_state_coverage',
        'n_raw_transitions':'pred_raw_transitions',
    })

    daily = true_daily[[
        'date','true_turn_on','true_turn_off','true_day_status',
        'true_always_on','true_always_off','true_already_on_at_04',
        'true_day_state_coverage','true_raw_transitions',
    ]].merge(
        pred_daily[[
            'date','pred_turn_on','pred_turn_off','pred_day_status',
            'pred_always_on','pred_always_off','pred_already_on_at_04',
            'pred_day_state_coverage','pred_raw_transitions',
        ]],
        on='date',
        how='outer',
    ).sort_values('date').reset_index(drop=True)

    daily['daily_eval_eligible'] = (
        pd.to_numeric(daily['true_day_state_coverage'], errors='coerce')
        >= CFG.min_truth_day_coverage
    ) & daily['true_day_status'].ne('INSUFFICIENT_DATA')

    both_on = (
        daily['daily_eval_eligible']
        & daily['true_turn_on'].notna()
        & daily['pred_turn_on'].notna()
    )
    both_off = (
        daily['daily_eval_eligible']
        & daily['true_turn_off'].notna()
        & daily['pred_turn_off'].notna()
    )

    daily['turn_on_error_min'] = np.nan
    daily['turn_off_error_min'] = np.nan

    daily.loc[both_on, 'turn_on_error_min'] = (
        pd.to_datetime(daily.loc[both_on, 'pred_turn_on'])
        - pd.to_datetime(daily.loc[both_on, 'true_turn_on'])
    ).dt.total_seconds() / 60.0

    daily.loc[both_off, 'turn_off_error_min'] = (
        pd.to_datetime(daily.loc[both_off, 'pred_turn_off'])
        - pd.to_datetime(daily.loc[both_off, 'true_turn_off'])
    ).dt.total_seconds() / 60.0

    daily['turn_on_detection_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_turn_on'].notna() == daily['pred_turn_on'].notna(),
        np.nan,
    )
    daily['turn_off_detection_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_turn_off'].notna() == daily['pred_turn_off'].notna(),
        np.nan,
    )
    daily['always_on_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_always_on'] == daily['pred_always_on'],
        np.nan,
    )
    daily['always_off_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_always_off'] == daily['pred_always_off'],
        np.nan,
    )
    return daily


In [ ]:
# CELL 14 — Held-out five-seed inference with fixed 0.5 ensemble threshold

INTERVAL_OUTPUTS = {}
POINTWISE_ROWS = []
DAILY_ROWS = []
MONTHLY_ROWS = []

folds = list(BUILDING_NAMES)
if CFG.max_folds is not None:
    folds = folds[:int(CFG.max_folds)]

for test_building in folds:
    train_buildings, val_building = fold_split(test_building)

    print('\n' + '=' * 100)
    print(
        'OUTER TEST:', test_building,
        '| TRAINING VALIDATION:', val_building,
        '| TRAIN:', train_buildings,
    )

    test_feat = FEATURE_TABLES[test_building].copy()

    if test_feat.empty:
        raise RuntimeError(
            f'{test_building}: empty held-out feature table.'
        )

    seed_scores = []

    for seed in CFG.lobo_seeds:
        model_path = MODEL_DIR / test_building / f'seed_{seed}_best.pt'
        if not model_path.exists():
            raise FileNotFoundError(f'Missing FINAL checkpoint: {model_path}')

        model, payload = load_model_checkpoint(model_path)

        # Strict split / final-stage audit.
        assert payload['test_building'] == test_building
        assert payload['val_building'] == val_building
        assert test_building not in payload['train_buildings']
        assert val_building not in payload['train_buildings']
        assert int(payload['best_stage']) == len(CFG.curriculum)

        seed_scores.append(
            predict_feature_frame(
                model,
                payload,
                test_feat,
            ).rename(str(seed))
        )

        del model, payload

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    score_df = pd.concat(seed_scores, axis=1)
    score_mean = score_df.mean(axis=1)
    score_std = score_df.std(axis=1, ddof=1)

    # Final decoder: one fixed threshold on the five-seed mean score.
    raw_state_05 = threshold_state(
        score_mean,
        CFG.decision_threshold,
    )
    predicted_state = clean_isolated_binary_flips(
        raw_state_05
    )

    # Seed-vote agreement is diagnostic only; it does not define the state.
    seed_binary = (score_df >= CFG.decision_threshold).astype(float)
    seed_valid_n = score_df.notna().sum(axis=1)
    seed_on_n = seed_binary.where(score_df.notna()).sum(axis=1)
    ensemble_agreement = pd.Series(
        np.nan,
        index=score_df.index,
        dtype=float,
    )
    valid_vote = seed_valid_n > 0
    ensemble_agreement.loc[valid_vote] = (
        np.maximum(
            seed_on_n.loc[valid_vote],
            seed_valid_n.loc[valid_vote] - seed_on_n.loc[valid_vote],
        )
        / seed_valid_n.loc[valid_vote]
    )

    out = test_feat[
        ['building','year','month','month_id','true_state']
    ].copy()

    out['score_mean'] = score_mean.reindex(out.index)
    out['score_std'] = score_std.reindex(out.index)
    out['decision_threshold'] = float(CFG.decision_threshold)
    out['raw_state_05'] = raw_state_05.reindex(out.index)
    out['predicted_state'] = predicted_state.reindex(out.index)
    out['ensemble_agreement'] = ensemble_agreement.reindex(out.index)

    # Backward-compatible alias used by some project code.
    out['oper_ensemble_state'] = out['predicted_state']

    INTERVAL_OUTPUTS[test_building] = out

    out.to_csv(
        PRED_DIR / f'{test_building}_interval_predictions.csv'
    )

    # --------------------------------------------------------------
    # Pointwise held-out state performance.
    # AUROC/AUPRC use the continuous ensemble mean score.
    # --------------------------------------------------------------
    valid = (
        out['true_state'].notna()
        & out['score_mean'].notna()
        & out['predicted_state'].notna()
    )

    y = out.loc[valid, 'true_state'].astype(int).to_numpy()
    score = out.loc[valid, 'score_mean'].to_numpy(float)
    pred = out.loc[valid, 'predicted_state'].astype(int).to_numpy()

    POINTWISE_ROWS.append({
        'building': test_building,
        'validation_building': val_building,
        'decision_threshold': float(CFG.decision_threshold),
        'n_intervals': int(len(y)),
        'accuracy': (
            float(accuracy_score(y, pred))
            if len(y) else np.nan
        ),
        'balanced_accuracy': (
            float(balanced_accuracy_score(y, pred))
            if len(y) and len(np.unique(y)) > 1 else np.nan
        ),
        'precision_on': (
            float(precision_score(y, pred, zero_division=0))
            if len(y) else np.nan
        ),
        'recall_on': (
            float(recall_score(y, pred, zero_division=0))
            if len(y) else np.nan
        ),
        'f1_on': (
            float(f1_score(y, pred, zero_division=0))
            if len(y) else np.nan
        ),
        'mcc': (
            float(matthews_corrcoef(y, pred))
            if len(y)
            and len(np.unique(y)) > 1
            and len(np.unique(pred)) > 1
            else np.nan
        ),
        'auroc': (
            float(roc_auc_score(y, score))
            if len(y) and len(np.unique(y)) > 1
            else np.nan
        ),
        'auprc': (
            float(average_precision_score(y, score))
            if len(y) and len(np.unique(y)) > 1
            else np.nan
        ),
        'mean_score_std': (
            float(out.loc[valid, 'score_std'].mean())
            if valid.any() else np.nan
        ),
        'mean_ensemble_agreement': (
            float(out.loc[valid, 'ensemble_agreement'].mean())
            if valid.any() else np.nan
        ),
    })

    # --------------------------------------------------------------
    # Daily operational schedule with corrected truth masking.
    # --------------------------------------------------------------
    true_state = TRUTH[test_building]['work']['binary_on']

    daily = build_daily_comparison(
        true_state,
        predicted_state,
    )

    daily['building'] = test_building
    daily['validation_building'] = val_building
    daily['decision_threshold'] = float(CFG.decision_threshold)
    daily['year'] = pd.to_datetime(daily['date']).dt.year
    daily['month'] = pd.to_datetime(daily['date']).dt.month

    DAILY_ROWS.extend(
        daily.to_dict('records')
    )

    # --------------------------------------------------------------
    # Monthly schedule characterization — truth-coverage safe.
    # --------------------------------------------------------------
    for (year, month), mdf in test_feat.groupby(['year','month']):
        year = int(year)
        month = int(month)

        d_all = daily.loc[
            (daily['year'] == year)
            & (daily['month'] == month)
        ].copy()

        d = d_all.loc[
            d_all['daily_eval_eligible'].fillna(False)
        ].copy()

        row = {
            'building': test_building,
            'year': year,
            'month': month,
            'month_id': f'{test_building}__{year:04d}-{month:02d}',
            'n_days_calendar_window': int(len(d_all)),
            'n_scored_days': int(len(d)),
            'decision_threshold': float(CFG.decision_threshold),
            'true_always_on_days': int(
                d['true_always_on'].fillna(False).sum()
            ),
            'pred_always_on_days': int(
                d['pred_always_on'].fillna(False).sum()
            ),
            'true_always_off_days': int(
                d['true_always_off'].fillna(False).sum()
            ),
            'pred_always_off_days': int(
                d['pred_always_off'].fillna(False).sum()
            ),
        }

        row.update(
            month_qc(
                mdf,
                true_state,
                year,
                month,
            )
        )

        row.update(
            event_detection_stats(
                d['true_turn_on'],
                d['pred_turn_on'],
                'turn_on',
            )
        )

        row.update(
            event_detection_stats(
                d['true_turn_off'],
                d['pred_turn_off'],
                'turn_off',
            )
        )

        row.update(
            matched_timing_stats(
                d['true_turn_on'],
                d['pred_turn_on'],
                'turn_on',
            )
        )

        row.update(
            matched_timing_stats(
                d['true_turn_off'],
                d['pred_turn_off'],
                'turn_off',
            )
        )

        row.update(
            schedule_distribution_stats(
                d['true_turn_on'],
                'true_on',
            )
        )
        row.update(
            schedule_distribution_stats(
                d['pred_turn_on'],
                'pred_on',
            )
        )
        row.update(
            schedule_distribution_stats(
                d['true_turn_off'],
                'true_off',
            )
        )
        row.update(
            schedule_distribution_stats(
                d['pred_turn_off'],
                'pred_off',
            )
        )

        for event in ['on','off']:
            true_med = row[
                f'true_{event}_median_min'
            ]
            pred_med = row[
                f'pred_{event}_median_min'
            ]
            true_iqr = row[
                f'true_{event}_iqr_min'
            ]
            pred_iqr = row[
                f'pred_{event}_iqr_min'
            ]

            row[
                f'turn_{event}_median_abs_error_min'
            ] = (
                abs(pred_med - true_med)
                if np.isfinite(pred_med)
                and np.isfinite(true_med)
                else np.nan
            )

            row[
                f'turn_{event}_iqr_abs_error_min'
            ] = (
                abs(pred_iqr - true_iqr)
                if np.isfinite(pred_iqr)
                and np.isfinite(true_iqr)
                else np.nan
            )

        MONTHLY_ROWS.append(row)

    print(
        f'{test_building}: held-out evaluation complete '
        f'(fixed threshold={CFG.decision_threshold:.2f})'
    )


pointwise_df = pd.DataFrame(POINTWISE_ROWS)
daily_schedule_df = pd.DataFrame(DAILY_ROWS)
monthly_schedule_df = pd.DataFrame(MONTHLY_ROWS)

pointwise_df.to_csv(
    TABLE_DIR / 'pointwise_metrics_by_building.csv',
    index=False,
)
daily_schedule_df.to_csv(
    TABLE_DIR / 'daily_operational_schedule.csv',
    index=False,
)
monthly_schedule_df.to_csv(
    TABLE_DIR / 'monthly_schedule_summary.csv',
    index=False,
)

print('POINTWISE PERFORMANCE — FIXED 0.5 ENSEMBLE THRESHOLD')
display(pointwise_df)

print('DAILY OPERATIONAL SCHEDULE')
display(daily_schedule_df.head(30))

print('MONTHLY SCHEDULE SUMMARY')
display(monthly_schedule_df)

In [ ]:
# CELL 15 — Journal-ready tables with truth-coverage-safe evaluation


def safe_binary_metrics(truth, pred, eval_mask=None):
    truth = pd.Series(truth)
    pred = pd.Series(pred)

    valid = truth.notna() & pred.notna()

    if eval_mask is not None:
        valid &= pd.Series(
            eval_mask
        ).fillna(False).to_numpy(dtype=bool)

    if not valid.any():
        return {
            'precision': np.nan,
            'recall': np.nan,
            'f1': np.nan,
            'accuracy': np.nan,
        }

    y = truth.loc[valid].astype(int)
    p = pred.loc[valid].astype(int)

    return {
        'precision': float(
            precision_score(y, p, zero_division=0)
        ),
        'recall': float(
            recall_score(y, p, zero_division=0)
        ),
        'f1': float(
            f1_score(y, p, zero_division=0)
        ),
        'accuracy': float(
            accuracy_score(y, p)
        ),
    }


# --------------------------------------------------------------
# Table 1 — Dataset / truth coverage
# --------------------------------------------------------------
dataset_rows = []

for b in BUILDING_NAMES:
    d = daily_schedule_df.loc[
        daily_schedule_df['building'] == b
    ].copy()

    m = d[
        'daily_eval_eligible'
    ].fillna(False)

    ds = d.loc[m]

    dataset_rows.append({
        'Building': b,
        'Analysis days': int(len(d)),
        'Scored truth days': int(m.sum()),
        'Scored-day fraction': (
            float(m.mean())
            if len(m) else np.nan
        ),
        'True valid ON events': int(
            ds['true_turn_on'].notna().sum()
        ),
        'True valid OFF events': int(
            ds['true_turn_off'].notna().sum()
        ),
        'True always ON days': int(
            ds['true_always_on'].fillna(False).sum()
        ),
        'True always OFF days': int(
            ds['true_always_off'].fillna(False).sum()
        ),
        'Mean true state coverage': float(
            d['true_day_state_coverage'].mean()
        ),
    })

Table1_Dataset = pd.DataFrame(
    dataset_rows
)


# --------------------------------------------------------------
# Table 2 — Pointwise ON/OFF inference
# --------------------------------------------------------------
Table2_Pointwise = pointwise_df[[
    'building',
    'validation_building',
    'decision_threshold',
    'n_intervals',
    'accuracy',
    'balanced_accuracy',
    'precision_on',
    'recall_on',
    'f1_on',
    'mcc',
    'auroc',
    'auprc',
    'mean_score_std',
    'mean_ensemble_agreement',
]].rename(columns={
    'building':'Building',
    'validation_building':'Training validation building',
    'decision_threshold':'Decision threshold',
    'n_intervals':'N intervals',
    'accuracy':'Accuracy',
    'balanced_accuracy':'Balanced accuracy',
    'precision_on':'ON precision',
    'recall_on':'ON recall',
    'f1_on':'ON F1',
    'mcc':'MCC',
    'auroc':'AUROC',
    'auprc':'AUPRC',
    'mean_score_std':'Mean seed-score SD',
    'mean_ensemble_agreement':'Mean seed-vote agreement',
})


# --------------------------------------------------------------
# Tables 3–5 — Daily operational performance
# --------------------------------------------------------------
EVENT_ROWS = []
TIMING_ROWS = []
STATUS_ROWS = []

for b in BUILDING_NAMES:
    d = daily_schedule_df.loc[
        daily_schedule_df['building'] == b
    ].copy()

    m = d[
        'daily_eval_eligible'
    ].fillna(False)

    on_det = event_detection_stats(
        d['true_turn_on'],
        d['pred_turn_on'],
        'on',
        m,
    )

    off_det = event_detection_stats(
        d['true_turn_off'],
        d['pred_turn_off'],
        'off',
        m,
    )

    on_t = matched_timing_stats(
        d['true_turn_on'],
        d['pred_turn_on'],
        'on',
        m,
    )

    off_t = matched_timing_stats(
        d['true_turn_off'],
        d['pred_turn_off'],
        'off',
        m,
    )

    EVENT_ROWS.append({
        'Building': b,
        'Scored days': int(m.sum()),
        'ON TP days': on_det['on_tp_days'],
        'ON false days': on_det['on_false_days'],
        'ON missed days': on_det['on_missed_days'],
        'ON precision': on_det['on_precision'],
        'ON recall': on_det['on_recall'],
        'ON F1': on_det['on_f1'],
        'OFF TP days': off_det['off_tp_days'],
        'OFF false days': off_det['off_false_days'],
        'OFF missed days': off_det['off_missed_days'],
        'OFF precision': off_det['off_precision'],
        'OFF recall': off_det['off_recall'],
        'OFF F1': off_det['off_f1'],
    })

    TIMING_ROWS.append({
        'Building': b,
        'ON matched days': on_t['on_matched_n'],
        'ON MAE [min]': on_t['on_mae_min'],
        'ON median AE [min]': on_t['on_median_ae_min'],
        'ON RMSE [min]': on_t['on_rmse_min'],
        'ON bias [min]': on_t['on_bias_min'],
        'ON q90 AE [min]': on_t['on_q90_ae_min'],
        'ON within ±15': on_t['on_within15'],
        'ON within ±30': on_t['on_within30'],
        'OFF matched days': off_t['off_matched_n'],
        'OFF MAE [min]': off_t['off_mae_min'],
        'OFF median AE [min]': off_t['off_median_ae_min'],
        'OFF RMSE [min]': off_t['off_rmse_min'],
        'OFF bias [min]': off_t['off_bias_min'],
        'OFF q90 AE [min]': off_t['off_q90_ae_min'],
        'OFF within ±15': off_t['off_within15'],
        'OFF within ±30': off_t['off_within30'],
    })

    aon = safe_binary_metrics(
        d['true_always_on'],
        d['pred_always_on'],
        m,
    )

    aoff = safe_binary_metrics(
        d['true_always_off'],
        d['pred_always_off'],
        m,
    )

    exact_status_acc = (
        float(
            (
                d.loc[m, 'true_day_status']
                == d.loc[m, 'pred_day_status']
            ).mean()
        )
        if m.any()
        else np.nan
    )

    STATUS_ROWS.append({
        'Building': b,
        'Exact day-status accuracy': exact_status_acc,
        'Always ON precision': aon['precision'],
        'Always ON recall': aon['recall'],
        'Always ON F1': aon['f1'],
        'Always OFF precision': aoff['precision'],
        'Always OFF recall': aoff['recall'],
        'Always OFF F1': aoff['f1'],
    })


Table3_EventDetection = pd.DataFrame(
    EVENT_ROWS
)

Table4_EventTiming = pd.DataFrame(
    TIMING_ROWS
)

Table5_DayStatus = pd.DataFrame(
    STATUS_ROWS
)


# --------------------------------------------------------------
# Table 6 — Monthly schedule characterization
# --------------------------------------------------------------
monthly_cols = [
    'building',
    'year',
    'month',
    'n_days_calendar_window',
    'n_scored_days',
    'decision_threshold',
    'feature_interval_coverage',
    'truth_interval_coverage',
    'complete_calendar_month',
    'coverage_eligible',
    'evaluation_eligible',
    'primary_evaluation_eligible',

    'true_on_n',
    'pred_on_n',
    'true_on_median_hhmm',
    'pred_on_median_hhmm',
    'true_on_iqr_min',
    'pred_on_iqr_min',

    'true_off_n',
    'pred_off_n',
    'true_off_median_hhmm',
    'pred_off_median_hhmm',
    'true_off_iqr_min',
    'pred_off_iqr_min',

    'true_always_on_days',
    'pred_always_on_days',
    'true_always_off_days',
    'pred_always_off_days',

    'turn_on_precision',
    'turn_on_recall',
    'turn_on_f1',
    'turn_on_mae_min',
    'turn_on_bias_min',
    'turn_on_within15',
    'turn_on_within30',

    'turn_off_precision',
    'turn_off_recall',
    'turn_off_f1',
    'turn_off_mae_min',
    'turn_off_bias_min',
    'turn_off_within15',
    'turn_off_within30',

    'turn_on_median_abs_error_min',
    'turn_on_iqr_abs_error_min',
    'turn_off_median_abs_error_min',
    'turn_off_iqr_abs_error_min',
]

Table6_MonthlySchedule = (
    monthly_schedule_df[
        monthly_cols
    ].copy()
)


# --------------------------------------------------------------
# Daily-status confusion matrix — scored truth days only
# --------------------------------------------------------------
scored_daily = daily_schedule_df.loc[
    daily_schedule_df[
        'daily_eval_eligible'
    ].fillna(False)
].copy()

day_status_confusion = pd.crosstab(
    scored_daily[
        'true_day_status'
    ],
    scored_daily[
        'pred_day_status'
    ],
    dropna=False,
)

day_status_confusion.to_csv(
    TABLE_DIR
    / 'Table_Daily_Status_Confusion.csv'
)


# --------------------------------------------------------------
# Export journal tables
# --------------------------------------------------------------
tables = {
    'Table1_Dataset':
        Table1_Dataset,

    'Table2_Pointwise':
        Table2_Pointwise,

    'Table3_EventDetection':
        Table3_EventDetection,

    'Table4_EventTiming':
        Table4_EventTiming,

    'Table5_DayStatus':
        Table5_DayStatus,

    'Table6_MonthlySchedule':
        Table6_MonthlySchedule,

    'TrainingWeightScheme':
        weight_scheme_table,
}

if 'training_summary_df' in globals():
    tables[
        'TrainingStage5Summary'
    ] = training_summary_df.copy()

for name, df in tables.items():
    df.to_csv(
        TABLE_DIR / f'{name}.csv',
        index=False,
    )


xlsx_path = (
    TABLE_DIR
    / 'Journal_Paper_Tables.xlsx'
)

try:
    with pd.ExcelWriter(
        xlsx_path,
        engine='openpyxl',
    ) as writer:

        for name, df in tables.items():
            df.to_excel(
                writer,
                sheet_name=name[:31],
                index=False,
            )

        day_status_confusion.to_excel(
            writer,
            sheet_name='DayStatusConfusion',
        )

    print(
        'Journal workbook:',
        xlsx_path,
    )

except Exception as exc:
    print(
        'Excel export skipped:',
        exc,
    )


for name, df in tables.items():
    print(
        '\n' + name
    )
    display(df)

print(
    '\nTruth-eligible daily status confusion'
)
display(
    day_status_confusion
)

In [ ]:
# CELL 16 — Journal-ready schedule and performance figures


def _clock_minutes(series):
    return np.array(
        [
            time_to_minutes(x)
            for x in series
        ],
        dtype=float,
    )


def _clock_axis(ax):
    ticks = np.arange(
        0,
        24 * 60 + 1,
        180,
    )

    ax.set_yticks(
        ticks
    )

    ax.set_yticklabels([
        f'{int(t//60):02d}:00'
        for t in ticks
    ])

    # Earliest time at the top.
    ax.set_ylim(
        24 * 60,
        0,
    )


def plot_monthly_schedule(
    building,
    year,
    month,
    show=True,
):
    d = daily_schedule_df.loc[
        (daily_schedule_df['building'] == building)
        & (daily_schedule_df['year'] == int(year))
        & (daily_schedule_df['month'] == int(month))
    ].copy()

    if d.empty:
        return None

    d = d.sort_values(
        'date'
    )

    day = pd.to_datetime(
        d['date']
    ).dt.day.to_numpy()

    true_on = _clock_minutes(
        d['true_turn_on']
    )
    pred_on = _clock_minutes(
        d['pred_turn_on']
    )
    true_off = _clock_minutes(
        d['true_turn_off']
    )
    pred_off = _clock_minutes(
        d['pred_turn_off']
    )

    fig, ax = plt.subplots(
        figsize=(12.5, 6.3)
    )

    # Low-truth-coverage days remain visible but are clearly unscored.
    for x, eligible in zip(
        day,
        d['daily_eval_eligible'].fillna(False),
    ):
        if not bool(eligible):
            ax.axvspan(
                x - 0.45,
                x + 0.45,
                alpha=0.08,
            )

    ax.scatter(
        day - 0.12,
        true_on,
        marker='o',
        s=42,
        label='True turn-on',
    )

    ax.scatter(
        day + 0.12,
        pred_on,
        marker='x',
        s=48,
        linewidths=1.6,
        label='Predicted turn-on',
    )

    ax.scatter(
        day - 0.12,
        true_off,
        marker='s',
        s=38,
        label='True turn-off',
    )

    ax.scatter(
        day + 0.12,
        pred_off,
        marker='+',
        s=58,
        linewidths=1.6,
        label='Predicted turn-off',
    )

    for (
        x,
        t_on,
        t_off,
        p_on,
        p_off,
        eligible,
    ) in zip(
        day,
        d['true_always_on'].fillna(False),
        d['true_always_off'].fillna(False),
        d['pred_always_on'].fillna(False),
        d['pred_always_off'].fillna(False),
        d['daily_eval_eligible'].fillna(False),
    ):
        if not bool(eligible):
            ax.text(
                x,
                35,
                'UNSCORED',
                ha='center',
                va='top',
                fontsize=6,
                rotation=90,
            )
            continue

        true_label = (
            'T:ON'
            if t_on
            else (
                'T:OFF'
                if t_off
                else ''
            )
        )

        pred_label = (
            'P:ON'
            if p_on
            else (
                'P:OFF'
                if p_off
                else ''
            )
        )

        if true_label:
            ax.text(
                x - 0.10,
                22,
                true_label,
                ha='center',
                va='top',
                fontsize=6.5,
                rotation=90,
            )

        if pred_label:
            ax.text(
                x + 0.10,
                75,
                pred_label,
                ha='center',
                va='top',
                fontsize=6.5,
                rotation=90,
            )

    _clock_axis(
        ax
    )

    ax.set_xlabel(
        'Day of month'
    )
    ax.set_ylabel(
        'Time of day'
    )

    ax.set_xticks(
        sorted(
            np.unique(day)
        )
    )

    ax.grid(
        alpha=0.2
    )

    ax.legend(
        ncol=2,
        loc='lower right',
    )

    ax.set_title(
        f'{building} — {year}-{month:02d}: '
        f'daily operational schedule '
        f'(ensemble threshold = {CFG.decision_threshold:.2f})'
    )

    fig.tight_layout()

    path = (
        PLOT_MONTHLY_DIR
        / f'{building}__{year}-{month:02d}__schedule.png'
    )

    fig.savefig(
        path,
        dpi=CFG.plot_dpi,
        bbox_inches='tight',
    )

    if show:
        plt.show()
    else:
        plt.close(fig)

    return path


# Save every building-month schedule figure.
if CFG.save_all_monthly_schedule_plots:
    for row in (
        monthly_schedule_df[
            ['building','year','month']
        ]
        .drop_duplicates()
        .itertuples(index=False)
    ):
        plot_monthly_schedule(
            row.building,
            int(row.year),
            int(row.month),
            show=False,
        )


# Display one representative primary-eligible month per building.
for b in BUILDING_NAMES:
    candidates = monthly_schedule_df.loc[
        (monthly_schedule_df['building'] == b)
        & monthly_schedule_df[
            'primary_evaluation_eligible'
        ].fillna(False)
    ].sort_values(
        ['year','month']
    )

    if not candidates.empty:
        r = candidates.iloc[0]

        plot_monthly_schedule(
            b,
            int(r['year']),
            int(r['month']),
            show=True,
        )


# --------------------------------------------------------------
# Figure: true vs predicted event-time scatter
# --------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.5, 5.5),
)

scored = daily_schedule_df.loc[
    daily_schedule_df[
        'daily_eval_eligible'
    ].fillna(False)
].copy()

for (
    ax,
    true_col,
    pred_col,
    title,
) in [
    (
        axes[0],
        'true_turn_on',
        'pred_turn_on',
        'Turn-on time',
    ),
    (
        axes[1],
        'true_turn_off',
        'pred_turn_off',
        'Turn-off time',
    ),
]:
    tmp = scored.loc[
        scored[true_col].notna()
        & scored[pred_col].notna()
    ]

    x = _clock_minutes(
        tmp[true_col]
    )
    y = _clock_minutes(
        tmp[pred_col]
    )

    ax.scatter(
        x,
        y,
        s=18,
        alpha=0.6,
    )

    if len(x):
        lo = min(
            np.nanmin(x),
            np.nanmin(y),
        )
        hi = max(
            np.nanmax(x),
            np.nanmax(y),
        )

        ax.plot(
            [lo,hi],
            [lo,hi],
            linestyle='--',
            linewidth=1.2,
            label='1:1',
        )
        ax.plot(
            [lo,hi],
            [lo+15,hi+15],
            linestyle=':',
            linewidth=1,
            alpha=0.7,
            label='±15 min',
        )
        ax.plot(
            [lo,hi],
            [lo-15,hi-15],
            linestyle=':',
            linewidth=1,
            alpha=0.7,
        )
        ax.plot(
            [lo,hi],
            [lo+30,hi+30],
            linestyle='-.',
            linewidth=0.8,
            alpha=0.5,
            label='±30 min',
        )
        ax.plot(
            [lo,hi],
            [lo-30,hi-30],
            linestyle='-.',
            linewidth=0.8,
            alpha=0.5,
        )

    ax.set_xlabel(
        'True time [min after midnight]'
    )
    ax.set_ylabel(
        'Predicted time [min after midnight]'
    )
    ax.set_title(
        title
    )
    ax.grid(
        alpha=0.2
    )
    ax.legend()

fig.tight_layout()

fig.savefig(
    PLOT_PAPER_DIR
    / 'Figure_EventTime_True_vs_Predicted.png',
    dpi=CFG.plot_dpi,
    bbox_inches='tight',
)

plt.show()


# --------------------------------------------------------------
# Figure: timing-error boxplots by building
# --------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5.3),
)

for (
    ax,
    error_col,
    title,
) in [
    (
        axes[0],
        'turn_on_error_min',
        'Turn-on timing error',
    ),
    (
        axes[1],
        'turn_off_error_min',
        'Turn-off timing error',
    ),
]:
    data = []

    for b in BUILDING_NAMES:
        arr = (
            pd.to_numeric(
                scored.loc[
                    scored['building'] == b,
                    error_col,
                ],
                errors='coerce',
            )
            .dropna()
            .to_numpy(float)
        )

        data.append(
            arr
        )

    ax.boxplot(
        data,
        labels=BUILDING_NAMES,
        showfliers=False,
    )

    ax.axhline(
        0,
        linestyle='--',
        linewidth=1,
    )

    ax.set_ylabel(
        'Prediction - truth [min]'
    )

    ax.set_title(
        title
    )

    ax.tick_params(
        axis='x',
        rotation=35,
    )

    ax.grid(
        axis='y',
        alpha=0.2,
    )

fig.tight_layout()

fig.savefig(
    PLOT_PAPER_DIR
    / 'Figure_Timing_Error_By_Building.png',
    dpi=CFG.plot_dpi,
    bbox_inches='tight',
)

plt.show()


# --------------------------------------------------------------
# Figure: event detection by building
# --------------------------------------------------------------
x = np.arange(
    len(Table3_EventDetection)
)

width = 0.12

fig, ax = plt.subplots(
    figsize=(13, 5.5)
)

for col, offset in [
    ('ON precision', -2.5),
    ('ON recall', -1.5),
    ('ON F1', -0.5),
    ('OFF precision', 0.5),
    ('OFF recall', 1.5),
    ('OFF F1', 2.5),
]:
    ax.bar(
        x + offset * width,
        Table3_EventDetection[
            col
        ].to_numpy(float),
        width,
        label=col,
    )

ax.set_xticks(
    x
)

ax.set_xticklabels(
    Table3_EventDetection[
        'Building'
    ],
    rotation=30,
)

ax.set_ylim(
    0,
    1.05,
)

ax.set_ylabel(
    'Score'
)

ax.set_title(
    'Daily operational event detection — fixed 0.5 ensemble threshold'
)

ax.grid(
    axis='y',
    alpha=0.2,
)

ax.legend(
    ncol=3
)

fig.tight_layout()

fig.savefig(
    PLOT_PAPER_DIR
    / 'Figure_Event_Detection_By_Building.png',
    dpi=CFG.plot_dpi,
    bbox_inches='tight',
)

plt.show()


# --------------------------------------------------------------
# Figure: pointwise ON/OFF performance
# --------------------------------------------------------------
x = np.arange(
    len(pointwise_df)
)

width = 0.22

fig, ax = plt.subplots(
    figsize=(11.5, 5.2)
)

ax.bar(
    x - width,
    pointwise_df[
        'f1_on'
    ],
    width,
    label='ON F1',
)

ax.bar(
    x,
    pointwise_df[
        'mcc'
    ],
    width,
    label='MCC',
)

ax.bar(
    x + width,
    pointwise_df[
        'balanced_accuracy'
    ],
    width,
    label='Balanced accuracy',
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    pointwise_df[
        'building'
    ],
    rotation=30,
)

ax.set_ylim(
    0,
    1.05,
)

ax.set_ylabel(
    'Score'
)

ax.set_title(
    'Pointwise chiller ON/OFF inference — fixed 0.5 ensemble threshold'
)

ax.grid(
    axis='y',
    alpha=0.2,
)

ax.legend(
    ncol=3
)

fig.tight_layout()

fig.savefig(
    PLOT_PAPER_DIR
    / 'Figure_Pointwise_Performance_By_Building.png',
    dpi=CFG.plot_dpi,
    bbox_inches='tight',
)

plt.show()


# --------------------------------------------------------------
# Figure: monthly schedule distribution errors
# Primary eligible complete months only.
# --------------------------------------------------------------
eligible_months = monthly_schedule_df.loc[
    monthly_schedule_df[
        'primary_evaluation_eligible'
    ].fillna(False)
].copy()

summary_month_errors = (
    eligible_months
    .groupby('building')
    .agg(
        ON_median_MAE=(
            'turn_on_median_abs_error_min',
            'mean',
        ),
        ON_IQR_MAE=(
            'turn_on_iqr_abs_error_min',
            'mean',
        ),
        OFF_median_MAE=(
            'turn_off_median_abs_error_min',
            'mean',
        ),
        OFF_IQR_MAE=(
            'turn_off_iqr_abs_error_min',
            'mean',
        ),
    )
    .reindex(
        BUILDING_NAMES
    )
    .reset_index()
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5.2),
)

x = np.arange(
    len(summary_month_errors)
)

width = 0.34

axes[0].bar(
    x - width/2,
    summary_month_errors[
        'ON_median_MAE'
    ],
    width,
    label='Median-time error',
)

axes[0].bar(
    x + width/2,
    summary_month_errors[
        'ON_IQR_MAE'
    ],
    width,
    label='IQR error',
)

axes[0].set_title(
    'Monthly turn-on schedule'
)

axes[1].bar(
    x - width/2,
    summary_month_errors[
        'OFF_median_MAE'
    ],
    width,
    label='Median-time error',
)

axes[1].bar(
    x + width/2,
    summary_month_errors[
        'OFF_IQR_MAE'
    ],
    width,
    label='IQR error',
)

axes[1].set_title(
    'Monthly turn-off schedule'
)

for ax in axes:
    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        summary_month_errors[
            'building'
        ],
        rotation=30,
    )

    ax.set_ylabel(
        'Absolute error [min]'
    )

    ax.grid(
        axis='y',
        alpha=0.2,
    )

    ax.legend()

fig.tight_layout()

fig.savefig(
    PLOT_PAPER_DIR
    / 'Figure_Monthly_Schedule_Distribution_Errors.png',
    dpi=CFG.plot_dpi,
    bbox_inches='tight',
)

plt.show()

print(
    'Paper-ready figures:',
    PLOT_PAPER_DIR,
)

print(
    'All monthly schedule figures:',
    PLOT_MONTHLY_DIR,
)

In [ ]:
# CELL 17 — Representative daily reconstruction figures + training curves


def choose_representative_dates(
    building,
    n=2,
):
    d = daily_schedule_df.loc[
        (daily_schedule_df['building'] == building)
        & daily_schedule_df[
            'daily_eval_eligible'
        ].fillna(False)
    ].copy()

    if d.empty:
        return []

    both = d.loc[
        d['true_turn_on'].notna()
        & d['true_turn_off'].notna()
    ]

    selected = []

    if not both.empty:
        picks = np.linspace(
            0,
            len(both) - 1,
            min(n, len(both)),
        ).round().astype(int)

        selected.extend(
            both.iloc[
                picks
            ]['date'].tolist()
        )

    if len(selected) < n:
        for date in d['date']:
            if date not in selected:
                selected.append(
                    date
                )

            if len(selected) >= n:
                break

    return selected[:n]


def plot_daily_reconstruction(
    building,
    date,
):
    cfg = BUILDINGS[
        building
    ]

    date = pd.Timestamp(
        date
    ).normalize()

    idx = pd.date_range(
        date,
        date
        + pd.Timedelta(
            hours=23,
            minutes=45,
        ),
        freq=f'{FREQ_MIN}min',
    )

    ptotal = pd.to_numeric(
        load_raw[
            cfg['total']
        ],
        errors='coerce',
    ).reindex(idx)

    pchiller = pd.to_numeric(
        load_raw[
            cfg['chiller']
        ],
        errors='coerce',
    ).reindex(idx)

    truth = TRUTH[
        building
    ][
        'work'
    ][
        'binary_on'
    ].reindex(idx)

    frame = INTERVAL_OUTPUTS[
        building
    ]

    pred = frame[
        'predicted_state'
    ].reindex(idx)

    raw = frame[
        'raw_state_05'
    ].reindex(idx)

    score = frame[
        'score_mean'
    ].reindex(idx)

    day = daily_schedule_df.loc[
        (
            daily_schedule_df[
                'building'
            ]
            == building
        )
        & (
            pd.to_datetime(
                daily_schedule_df[
                    'date'
                ]
            ).dt.normalize()
            == date
        )
    ]

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(12.5, 8),
        sharex=True,
        gridspec_kw={
            'height_ratios':
                [2.2, 1, 1]
        },
    )

    axes[0].plot(
        idx,
        ptotal,
        label='Total building load',
        linewidth=1.4,
    )

    axes[0].plot(
        idx,
        pchiller,
        label='Measured chiller load',
        linewidth=1.3,
    )

    axes[0].set_ylabel(
        'Power [kW]'
    )

    axes[0].grid(
        alpha=0.2
    )

    axes[0].legend(
        loc='upper right'
    )

    axes[1].step(
        idx,
        truth,
        where='post',
        label='True state',
        linewidth=2,
    )

    axes[1].step(
        idx,
        raw,
        where='post',
        label='Raw score ≥ 0.5',
        linewidth=1.0,
        alpha=0.55,
    )

    axes[1].step(
        idx,
        pred,
        where='post',
        label='Final predicted state',
        linewidth=1.8,
    )

    axes[1].set_ylim(
        -0.15,
        1.15,
    )

    axes[1].set_yticks(
        [0,1]
    )

    axes[1].set_yticklabels([
        'OFF',
        'ON',
    ])

    axes[1].set_ylabel(
        'State'
    )

    axes[1].grid(
        alpha=0.2
    )

    axes[1].legend(
        loc='upper right'
    )

    axes[2].plot(
        idx,
        score,
        label='Five-seed mean ON score',
        linewidth=1.4,
    )

    axes[2].axhline(
        CFG.decision_threshold,
        linestyle='--',
        linewidth=1.2,
        label=f'Fixed threshold = {CFG.decision_threshold:.2f}',
    )

    axes[2].set_ylim(
        -0.05,
        1.05,
    )

    axes[2].set_ylabel(
        'ON score'
    )

    axes[2].grid(
        alpha=0.2
    )

    axes[2].legend(
        loc='upper right'
    )

    if not day.empty:
        r = day.iloc[0]

        markers = [
            (
                r['true_turn_on'],
                ':',
            ),
            (
                r['pred_turn_on'],
                '--',
            ),
            (
                r['true_turn_off'],
                ':',
            ),
            (
                r['pred_turn_off'],
                '--',
            ),
        ]

        for ts, ls in markers:
            if pd.notna(ts):
                for ax in axes:
                    ax.axvline(
                        pd.Timestamp(ts),
                        linestyle=ls,
                        linewidth=1.2,
                        alpha=0.8,
                    )

        scored_text = (
            'scored'
            if bool(
                r[
                    'daily_eval_eligible'
                ]
            )
            else
            'truth coverage insufficient — unscored'
        )
    else:
        scored_text = ''

    axes[0].set_title(
        f'{building} — {date.date()} — {scored_text}'
    )

    fig.tight_layout()

    return fig


for b in BUILDING_NAMES:
    for d in choose_representative_dates(
        b,
        CFG.representative_days_per_building,
    ):
        fig = plot_daily_reconstruction(
            b,
            d,
        )

        fig.savefig(
            PLOT_DAILY_DIR
            / f'{b}__{pd.Timestamp(d).date()}__daily_reconstruction.png',
            dpi=CFG.plot_dpi,
            bbox_inches='tight',
        )

        plt.show()


# --------------------------------------------------------------
# Training curves
# --------------------------------------------------------------
for b in folds:
    for seed in CFG.lobo_seeds:
        hp = (
            MODEL_DIR
            / b
            / f'seed_{seed}_history.csv'
        )

        if not hp.exists():
            continue

        h = pd.read_csv(
            hp
        )

        fig, ax = plt.subplots(
            figsize=(9, 4.5)
        )

        ax.plot(
            h['epoch'],
            h['train_fixed_mse'],
            label='Train fixed event-aware MSE',
        )

        ax.plot(
            h['epoch'],
            h['val_fixed_mse'],
            label='Validation fixed event-aware MSE',
        )

        ax.plot(
            h['epoch'],
            h['train_optimization_mse'],
            alpha=0.55,
            label='Stage optimization MSE',
        )

        for st in sorted(
            h['stage'].unique()
        ):
            ax.axvline(
                int(
                    h.loc[
                        h['stage'] == st,
                        'epoch',
                    ].min()
                ),
                linestyle=':',
                alpha=0.3,
            )

        ax.set_xlabel(
            'Epoch'
        )

        ax.set_ylabel(
            'Weighted MSE'
        )

        ax.set_title(
            f'{b} — seed {seed}'
        )

        ax.grid(
            alpha=0.2
        )

        ax.legend()

        fig.tight_layout()

        fig.savefig(
            PLOT_TRAIN_DIR
            / f'{b}__seed_{seed}__training.png',
            dpi=CFG.plot_dpi,
            bbox_inches='tight',
        )

        plt.close(
            fig
        )

print(
    'Daily reconstruction figures:',
    PLOT_DAILY_DIR,
)

print(
    'Training curves:',
    PLOT_TRAIN_DIR,
)

In [ ]:
# CELL 18 — Global paper summary: final fixed 0.5 decoder

scored_mask = daily_schedule_df[
    'daily_eval_eligible'
].fillna(False)

primary_month_mask = monthly_schedule_df[
    'primary_evaluation_eligible'
].fillna(False)

overall_on_det = event_detection_stats(
    daily_schedule_df['true_turn_on'],
    daily_schedule_df['pred_turn_on'],
    'event',
    scored_mask,
)

overall_on_time = matched_timing_stats(
    daily_schedule_df['true_turn_on'],
    daily_schedule_df['pred_turn_on'],
    'event',
    scored_mask,
)

overall_off_det = event_detection_stats(
    daily_schedule_df['true_turn_off'],
    daily_schedule_df['pred_turn_off'],
    'event',
    scored_mask,
)

overall_off_time = matched_timing_stats(
    daily_schedule_df['true_turn_off'],
    daily_schedule_df['pred_turn_off'],
    'event',
    scored_mask,
)

pooled_always_on = safe_binary_metrics(
    daily_schedule_df['true_always_on'],
    daily_schedule_df['pred_always_on'],
    scored_mask,
)

pooled_always_off = safe_binary_metrics(
    daily_schedule_df['true_always_off'],
    daily_schedule_df['pred_always_off'],
    scored_mask,
)

exact_status_accuracy = (
    float(
        (
            daily_schedule_df.loc[
                scored_mask,
                'true_day_status',
            ]
            == daily_schedule_df.loc[
                scored_mask,
                'pred_day_status',
            ]
        ).mean()
    )
    if scored_mask.any()
    else np.nan
)

GLOBAL_SUMMARY = pd.DataFrame([
    {
        'Metric':
            'Fixed ensemble decision threshold',
        'Value':
            float(CFG.decision_threshold),
    },
    {
        'Metric':
            'Truth-eligible scored days',
        'Value':
            int(scored_mask.sum()),
    },
    {
        'Metric':
            'Mean pointwise ON F1 across buildings',
        'Value':
            float(pointwise_df['f1_on'].mean()),
    },
    {
        'Metric':
            'Mean pointwise MCC across buildings',
        'Value':
            float(pointwise_df['mcc'].mean()),
    },
    {
        'Metric':
            'Pooled turn-on precision',
        'Value':
            overall_on_det['event_precision'],
    },
    {
        'Metric':
            'Pooled turn-on recall',
        'Value':
            overall_on_det['event_recall'],
    },
    {
        'Metric':
            'Pooled turn-on detection F1',
        'Value':
            overall_on_det['event_f1'],
    },
    {
        'Metric':
            'Pooled turn-on timing MAE [min]',
        'Value':
            overall_on_time['event_mae_min'],
    },
    {
        'Metric':
            'Pooled turn-on median AE [min]',
        'Value':
            overall_on_time['event_median_ae_min'],
    },
    {
        'Metric':
            'Pooled turn-on bias [min]',
        'Value':
            overall_on_time['event_bias_min'],
    },
    {
        'Metric':
            'Pooled turn-on within ±15 min',
        'Value':
            overall_on_time['event_within15'],
    },
    {
        'Metric':
            'Pooled turn-on within ±30 min',
        'Value':
            overall_on_time['event_within30'],
    },
    {
        'Metric':
            'Pooled turn-off precision',
        'Value':
            overall_off_det['event_precision'],
    },
    {
        'Metric':
            'Pooled turn-off recall',
        'Value':
            overall_off_det['event_recall'],
    },
    {
        'Metric':
            'Pooled turn-off detection F1',
        'Value':
            overall_off_det['event_f1'],
    },
    {
        'Metric':
            'Pooled turn-off timing MAE [min]',
        'Value':
            overall_off_time['event_mae_min'],
    },
    {
        'Metric':
            'Pooled turn-off median AE [min]',
        'Value':
            overall_off_time['event_median_ae_min'],
    },
    {
        'Metric':
            'Pooled turn-off bias [min]',
        'Value':
            overall_off_time['event_bias_min'],
    },
    {
        'Metric':
            'Pooled turn-off within ±15 min',
        'Value':
            overall_off_time['event_within15'],
    },
    {
        'Metric':
            'Pooled turn-off within ±30 min',
        'Value':
            overall_off_time['event_within30'],
    },
    {
        'Metric':
            'Pooled always-ON F1',
        'Value':
            pooled_always_on['f1'],
    },
    {
        'Metric':
            'Pooled always-OFF F1',
        'Value':
            pooled_always_off['f1'],
    },
    {
        'Metric':
            'Exact daily-status accuracy',
        'Value':
            exact_status_accuracy,
    },
    {
        'Metric':
            'Primary eligible complete building-months',
        'Value':
            int(primary_month_mask.sum()),
    },
    {
        'Metric':
            'Mean monthly turn-on median-time error [min]',
        'Value':
            float(
                monthly_schedule_df.loc[
                    primary_month_mask,
                    'turn_on_median_abs_error_min',
                ].mean()
            ),
    },
    {
        'Metric':
            'Mean monthly turn-off median-time error [min]',
        'Value':
            float(
                monthly_schedule_df.loc[
                    primary_month_mask,
                    'turn_off_median_abs_error_min',
                ].mean()
            ),
    },
])

GLOBAL_SUMMARY.to_csv(
    TABLE_DIR
    / 'Table_Global_Summary.csv',
    index=False,
)

display(
    GLOBAL_SUMMARY
)

In [ ]:
# CELL 19 — Integrity checks and final paper run manifest

# --------------------------------------------------------------
# Feature leakage audit
# --------------------------------------------------------------
for forbidden in [
    'chiller',
    'sttr',
    'reference_state',
    'binary_on',
    'primary_event',
    'secondary_transition',
    'building_id',
]:
    assert not any(
        forbidden.lower()
        in f.lower()
        for f in FEATURE_NAMES
    ), (
        forbidden,
        FEATURE_NAMES,
    )

assert len(
    FEATURE_NAMES
) == 41

assert (
    'is_weekend'
    in FEATURE_NAMES
)

assert (
    'tod_sin'
    in FEATURE_NAMES
)

assert (
    'tod_cos'
    in FEATURE_NAMES
)


# --------------------------------------------------------------
# Event weighting hierarchy
# --------------------------------------------------------------
assert (
    CFG.fixed_weight_primary_exact
    > CFG.fixed_weight_secondary_exact
    > CFG.fixed_weight_ordinary
)

assert (
    CFG.fixed_weight_primary_pm15
    > CFG.fixed_weight_secondary_pm15
    > CFG.fixed_weight_ordinary
)

assert (
    CFG.fixed_weight_primary_pm30
    > CFG.fixed_weight_secondary_pm30
    >= CFG.fixed_weight_ordinary
)


# --------------------------------------------------------------
# Operational-event semantics
# --------------------------------------------------------------
assert (
    CFG.event_pre_window_min
    == 60
)

assert (
    CFG.event_pre_dominance_ratio
    == 0.75
)

assert (
    CFG.event_post_steps
    == 2
)

assert (
    len(CFG.lobo_seeds)
    == 5
)


# --------------------------------------------------------------
# Final decoder audit
# --------------------------------------------------------------
assert (
    float(CFG.decision_threshold)
    == 0.50
)

assert (
    'decision_threshold'
    in pointwise_df.columns
)

assert (
    np.allclose(
        pointwise_df[
            'decision_threshold'
        ].to_numpy(float),
        0.50,
    )
)

assert (
    'theta_on'
    not in pointwise_df.columns
)

assert (
    'theta_off'
    not in pointwise_df.columns
)


# --------------------------------------------------------------
# Stage-5-guaranteed curriculum audit
# --------------------------------------------------------------
assert (
    len(CFG.curriculum)
    == 5
)

assert (
    CFG.max_epochs
    == 500
)

assert (
    CFG.min_epochs_per_stage
    == 8
)

assert (
    CFG.max_epochs_per_stage
    == 60
)

assert (
    CFG.train_plateau_patience
    == 6
)

assert (
    CFG.val_rise_patience
    == 5
)

assert (
    CFG.final_stage_patience
    == 10
)

assert (
    CFG.smoothing_window
    == 3
)

if 'training_summary_df' in globals():
    assert not training_summary_df.empty

    assert (
        training_summary_df[
            'best_stage'
        ]
        == len(
            CFG.curriculum
        )
    ).all()


# --------------------------------------------------------------
# Evaluation coverage audit
# --------------------------------------------------------------
assert (
    'daily_eval_eligible'
    in daily_schedule_df.columns
)

assert (
    'truth_interval_coverage'
    in monthly_schedule_df.columns
)

assert (
    'primary_evaluation_eligible'
    in monthly_schedule_df.columns
)

assert (
    monthly_schedule_df.loc[
        monthly_schedule_df[
            'primary_evaluation_eligible'
        ].fillna(False),
        'truth_interval_coverage',
    ]
    >= CFG.min_month_truth_coverage
).all()

assert (
    monthly_schedule_df.loc[
        monthly_schedule_df[
            'primary_evaluation_eligible'
        ].fillna(False),
        'feature_interval_coverage',
    ]
    >= CFG.min_month_interval_coverage
).all()

assert (
    monthly_schedule_df.loc[
        monthly_schedule_df[
            'primary_evaluation_eligible'
        ].fillna(False),
        'complete_calendar_month',
    ]
).all()


# --------------------------------------------------------------
# Obsolete classifications must not exist
# --------------------------------------------------------------
for df_name, df in [
    (
        'daily_schedule_df',
        daily_schedule_df,
    ),
    (
        'monthly_schedule_df',
        monthly_schedule_df,
    ),
]:
    for obsolete in [
        'ABC',
        'final_category',
        'start_pattern',
    ]:
        assert not any(
            obsolete.lower()
            in str(col).lower()
            for col in df.columns
        ), (
            df_name,
            obsolete,
        )


manifest = {
    'framework':
        'Event-aware daily chiller schedule inference with fixed 0.5 ensemble threshold',

    'building_set':
        list(
            BUILDING_NAMES
        ),

    'final_objective': [
        'daily turn-on detection and time',
        'daily turn-off detection and time',
        'always-ON day detection',
        'always-OFF day detection',
        'monthly start/stop schedule distributions',
    ],

    'removed_objectives': [
        'A/B/C monthly categorization',
        'Fixed/Semi-dynamic/Dynamic monthly category',
        'seven-class final archetype',
        'post-training threshold calibration',
        'hysteresis decoding',
    ],

    'feature_names':
        FEATURE_NAMES,

    'n_features':
        len(
            FEATURE_NAMES
        ),

    'event_rule': {
        'pre_window_min':
            CFG.event_pre_window_min,

        'pre_dominance_ratio':
            CFG.event_pre_dominance_ratio,

        'pre_min_valid_fraction':
            CFG.event_pre_min_valid_fraction,

        'post_steps':
            CFG.event_post_steps,

        'morning_turn_on_search': [
            CFG.morning_start,
            CFG.morning_end,
        ],

        'principal_turn_on':
            'first validated morning OFF->ON',

        'principal_turn_off':
            'last validated ON->OFF after principal start; after 04:00 only if already ON',
    },

    'decoder': {
        'score':
            'mean of five independently trained Stage-5 seed scores',

        'decision_threshold':
            float(
                CFG.decision_threshold
            ),

        'rule':
            'ON iff ensemble mean score >= 0.5',

        'isolated_flip_cleanup':
            True,

        'post_training_calibration':
            False,

        'hysteresis':
            False,
    },

    'evaluation_masking': {
        'min_truth_day_coverage':
            CFG.min_truth_day_coverage,

        'min_feature_month_coverage':
            CFG.min_month_interval_coverage,

        'min_truth_month_coverage':
            CFG.min_month_truth_coverage,

        'primary_month_requires_complete_calendar_month':
            True,
    },

    'fixed_training_weights': {
        'ordinary':
            CFG.fixed_weight_ordinary,

        'secondary_pm30':
            CFG.fixed_weight_secondary_pm30,

        'secondary_pm15':
            CFG.fixed_weight_secondary_pm15,

        'secondary_exact':
            CFG.fixed_weight_secondary_exact,

        'primary_pm30':
            CFG.fixed_weight_primary_pm30,

        'primary_pm15':
            CFG.fixed_weight_primary_pm15,

        'primary_exact':
            CFG.fixed_weight_primary_exact,
    },

    'curriculum':
        CFG.curriculum,

    'curriculum_policy': {
        'warmup_stages':
            [1, 2, 3, 4],

        'final_selection_stage':
            5,

        'all_seeds_must_reach_stage5':
            True,

        'max_epochs_per_warmup_stage':
            CFG.max_epochs_per_stage,

        'final_stage_patience':
            CFG.final_stage_patience,
    },

    'ensemble_seeds':
        CFG.lobo_seeds,

    'checkpoint_reuse': {
        'enabled':
            CFG.reuse_previous_compatible_checkpoints,

        'previous_compatible_result_dir':
            CFG.previous_compatible_result_dir,

        'compatibility_is_rechecked_by_training_logic':
            True,
    },

    'outputs': {
        'daily_schedule':
            str(
                TABLE_DIR
                / 'daily_operational_schedule.csv'
            ),

        'monthly_schedule':
            str(
                TABLE_DIR
                / 'monthly_schedule_summary.csv'
            ),

        'pointwise_metrics':
            str(
                TABLE_DIR
                / 'pointwise_metrics_by_building.csv'
            ),

        'global_summary':
            str(
                TABLE_DIR
                / 'Table_Global_Summary.csv'
            ),

        'journal_tables_workbook':
            str(
                TABLE_DIR
                / 'Journal_Paper_Tables.xlsx'
            ),

        'paper_figures':
            str(
                PLOT_PAPER_DIR
            ),

        'monthly_schedule_figures':
            str(
                PLOT_MONTHLY_DIR
            ),

        'daily_reconstruction_figures':
            str(
                PLOT_DAILY_DIR
            ),
    },
}

with open(
    RESULT_DIR
    / 'Run_Manifest_FINAL_PAPER.json',
    'w',
    encoding='utf-8',
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        default=str,
    )

print(
    json.dumps(
        manifest,
        indent=2,
        default=str,
    )
)

## Reproducibility checks

The existing code checks feature integrity, held-out model selection, fixed-threshold decoding, and eligibility of scored periods. The experimental protocol uses **41 input features**, **seven outer folds**, and **five seeds per fold**. No result artifacts, plots, or notebook execution traces are stored in this release.

Run `02_validation_extensions.ipynb` only after the main analysis has produced its local checkpoints and evaluation artifacts.